***Imprting libraries and Understanding data***

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
from shapely.geometry import Point, box
from shapely.geometry.polygon import Polygon
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.patches as mpatches

csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
gda_updated_15jan2026 = pd.read_csv(csv_path)

In [ ]:
gda_updated_15jan2026.head()

In [ ]:
gda_updated_15jan2026.columns

In [ ]:
print("Total complaints:", len(gda_updated_15jan2026))
gda_updated_15jan2026["Status"].value_counts()

In [ ]:
# Count missing values (NaN)
empty_offences = gda_updated_15jan2026["Offences"].isna().sum()

print("Number of empty rows in Offences column:", empty_offences)


***Table- Offence-Complaint count***

In [ ]:
offence_count_table = (
    gda_updated_15jan2026
    .groupby("Offences")["Compliant ID"]
    .count()
    .reset_index(name="Complaint Count")
    .sort_values(by="Complaint Count", ascending=False)
    .reset_index(drop=True))

offence_count_table


***Table- Offence-Status count-Complaint count***

In [ ]:
import pandas as pd

# Ensure Status values are consistent (optional but recommended)
gda_updated_15jan2026["Status"] = (gda_updated_15jan2026["Status"].str.strip().str.title())

# Create pivot table
offence_status_table = (gda_updated_15jan2026
    .pivot_table(index="Offences", columns="Status", values="Compliant ID", aggfunc="count", fill_value=0).reset_index())

# Keep only required status columns (add missing ones if absent)
required_status = ["Resolved", "Pending", "Rejected", "Unrelated"]
for col in required_status:
    if col not in offence_status_table.columns:
        offence_status_table[col] = 0

# Reorder columns
offence_status_table = offence_status_table[
    ["Offences", "Resolved", "Pending", "Rejected", "Unrelated"]]

# Add Total column
offence_status_table["Total"] = (offence_status_table[["Resolved", "Pending", "Rejected", "Unrelated"]].sum(axis=1))

# Sort offences by total complaints (max on top)
offence_status_table = offence_status_table.sort_values(
    by="Total", ascending=False
).reset_index(drop=True)

offence_status_table

***Table- Offence-Status count-Complaint count- with %***

In [ ]:
# Ensure Status values are consistent
gda_updated_15jan2026["Status"] = gda_updated_15jan2026["Status"].str.strip().str.title()

# Create pivot table
offence_status_table = gda_updated_15jan2026.pivot_table(
    index="Offences",
    columns="Status",
    values="Compliant ID",
    aggfunc="count",
    fill_value=0).reset_index()

# Ensure all required status columns exist
required_status = ["Resolved", "Pending", "Rejected", "Unrelated"]
for col in required_status:
    if col not in offence_status_table.columns:
        offence_status_table[col] = 0

# Reorder columns
offence_status_table = offence_status_table[
    ["Offences"] + required_status]

# Add Total column
offence_status_table["Total"] = offence_status_table[required_status].sum(axis=1)

# Add percentage columns
for col in required_status:
    offence_status_table[f"{col} %"] = (offence_status_table[col] / offence_status_table["Total"] * 100).round(2)

# Sort by Total
offence_status_table = offence_status_table.sort_values(by="Total", ascending=False).reset_index(drop=True)

offence_status_table


***Graph- top Offence-Complaint count***

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Offence-wise count
offence_counts = (
    gda_updated_15jan2026.groupby("Offences")["Compliant ID"].count().reset_index(name="Count of cases").sort_values("Count of cases", ascending=False))

# 2. Top 4 offences + Others
top4 = offence_counts.head(4)

others = pd.DataFrame({
    "Offences": ["Others"],
    "Count of cases": [offence_counts.iloc[4:]["Count of cases"].sum()]})

final_df = pd.concat([top4, others], ignore_index=True)

# 3. EXACT X-axis names (match image)
x_axis_names = {
    "Illegal dumping of Garbage on road sides/ vacant land": "Illegal dumping",
    "Potholes on road sides": "Potholes on Roads",
    "Road Dust": "Road Dust",
    "Dumping of Construction & Demolition Waste": "Dumping of C&D",
    "Others": "Others"}

final_df["Offences"] = final_df["Offences"].replace(x_axis_names)

# 4. Plot
plt.figure(figsize=(6, 5))
bars = plt.bar(final_df["Offences"], final_df["Count of cases"], color="#029bd6")

plt.ylabel("Count of cases")
plt.xlabel("")

# 5. Vertical X-axis labels (like image)
plt.xticks(rotation=90, fontsize=10)

# 6. Vertical values inside bars
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, h * 0.95, f"{int(h)}", ha="center", va="top", rotation=90, fontsize=9, color="white")

plt.tight_layout()
plt.show()


***Graph- top Offence with Complaint count (Pendind-Resolved)***

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Prepare data
df = gda_updated_15jan2026.copy()

# Group by Offence and Status
offence_status = (
    df.groupby(["Offences", "Status"])["Compliant ID"]
    .count()
    .unstack(fill_value=0)
    .reset_index()
)

# Keep only required statuses
offence_status = offence_status[["Offences", "Resolved", "Pending"]]

# Total cases
offence_status["Total"] = offence_status["Resolved"] + offence_status["Pending"]

# Sort by total cases (descending)
offence_status = offence_status.sort_values("Total", ascending=False)

# 2. Top 4 offences + Others
top4 = offence_status.head(4)

others = pd.DataFrame({
    "Offences": ["Others"],
    "Resolved": [offence_status.iloc[4:]["Resolved"].sum()],
    "Pending":  [offence_status.iloc[4:]["Pending"].sum()],
})

final_df = pd.concat([top4, others], ignore_index=True)

# 3. Short offence names
offence_short_names = {
    "Illegal dumping of Garbage on road sides/ vacant land": "Illegal dumping",
    "Potholes on road sides": "Potholes on Roads",
    "Road Dust": "Road Dust",
    "Dumping of Construction & Demolition Waste": "Dumping of C&D",
    "Others": "Others"
}

final_df["Offences"] = final_df["Offences"].replace(offence_short_names)

# 4. Percentage calculation
resolved = final_df["Resolved"].values
pending  = final_df["Pending"].values
total    = resolved + pending

resolved_pct = resolved / total * 100
pending_pct  = pending / total * 100

x = np.arange(len(final_df))
bar_width = 0.6

# 5. Plot stacked bar chart
fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(x, pending_pct, width=bar_width, color="#e85724", label="Pending")
ax.bar(x, resolved_pct, width=bar_width, bottom=pending_pct,
       color="#8cb73f", label="Resolved")

# Percentage labels inside bars
for i in range(len(x)):
    ax.text(
        x[i],
        pending_pct[i] / 2,
        f"{pending_pct[i]:.2f}%",
        ha="center", va="center", fontsize=9, color="white"
    )
    ax.text(
        x[i],
        pending_pct[i] + resolved_pct[i] / 2,
        f"{resolved_pct[i]:.2f}%",
        ha="center", va="center", fontsize=9, color="white"
    )

# Axis formatting
ax.set_xticks(x)
ax.set_xticklabels(final_df["Offences"])
ax.set_ylabel("Percentage of complaints")
ax.set_ylim(0, 100)
ax.set_yticks([0, 20, 40, 60, 80, 100])
ax.set_yticklabels(["0%", "20%", "40%", "60%", "80%", "100%"])

ax.legend(loc="upper left", frameon=False)

# 6. Table below the chart
table_data = [
    [f"{v}" for v in resolved],
    [f"{v}" for v in pending]
]

row_labels = ["Resolved", "Pending"]

table = plt.table(
    cellText=table_data,
    rowLabels=row_labels,
    colLabels=final_df["Offences"],
    cellLoc="center",
    rowLoc="center",
    loc="bottom",
    bbox=[0.0, -0.38, 1.0, 0.25]
)

plt.subplots_adjust(bottom=0.32)
plt.show()


***Table- Department-Complaint count-Status count- with %***

In [ ]:
# Ensure Status values are consistent
gda_updated_15jan2026["Status"] = (gda_updated_15jan2026["Status"].str.strip().str.title())

# Create department-wise status table
department_count_table = (
    gda_updated_15jan2026
    .pivot_table(
        index="Department Name",
        columns="Status",
        values="Compliant ID",
        aggfunc="count",
        fill_value=0).reset_index())

# Keep required status columns (add if missing)
required_cols = ["Resolved", "Pending", "Unrelated", "Rejected"]
for col in required_cols:
    if col not in department_count_table.columns:
        department_count_table[col] = 0

# Add total complaint count
department_count_table["Complaint Count"] = (
    department_count_table[required_cols].sum(axis=1))

# Percentage columns (row-wise)
for col in required_cols:
    department_count_table[f"{col} %"] = (
        department_count_table[col] / department_count_table["Complaint Count"] * 100).round(2)

# Reorder columns
department_count_table = department_count_table[
    ["Department Name", "Complaint Count",
     "Resolved", "Resolved %",
     "Pending", "Pending %",
     "Unrelated", "Unrelated %",
     "Rejected", "Rejected %"]]

# Sort by total complaints
department_count_table = department_count_table.sort_values(
    "Complaint Count", ascending=False)

department_count_table


***Graph- top Department with Complaint count***

In [ ]:
# 1. Department-wise complaint count
dept_counts = (
    gda_updated_15jan2026.groupby("Department Name")["Compliant ID"].count().reset_index(name="Complaint Count")
    .sort_values("Complaint Count", ascending=False))

# 2. Select top 5 departments
top5 = dept_counts.head(5)

# 3. Combine remaining departments into "Others"
others = pd.DataFrame({
    "Department Name": ["Others"],
    "Complaint Count": [dept_counts.iloc[5:]["Complaint Count"].sum()]})

final_df = pd.concat([top5, others], ignore_index=True)

# 4. Shorten department names (explicit mapping)
dept_short_names = {
    "Municipal Corporation of Delhi (MCD)": "MCD",
    "Public Works Department (PWD)": "PWD",
    "Delhi Development Authority (DDA)": "DDA",
    "Delhi Jal Board (DJB)": "DJB",
    "Irrigation & Flood Control (IFC)": "IFC",
    "Others": "Others"}

final_df["Department Name"] = final_df["Department Name"].replace(dept_short_names)

# 5. Plot
plt.figure(figsize=(6, 5))
bars = plt.bar( final_df["Department Name"], final_df["Complaint Count"], color="#029bd6")

plt.ylabel("Complaint Count")
plt.xlabel("Department")

# 6. Vertical value labels inside bars
for bar in bars:
    h = bar.get_height()
    plt.text( bar.get_x() + bar.get_width() / 2, h * 0.95, f"{int(h)}", ha="center", va="top", rotation=90, fontsize=9, color="white")

plt.tight_layout()
plt.show()


***Graph- top Department with Complaint count (Pendind-Resolved)***

In [ ]:
# Ensure clean Status values
gda_updated_15jan2026["Status"] = (gda_updated_15jan2026["Status"].str.strip().str.title())

# 1. Department-wise status counts (Resolved & Pending only)
dept_status = (
    gda_updated_15jan2026
    .pivot_table(
        index="Department Name",
        columns="Status",
        values="Compliant ID",
        aggfunc="count",
        fill_value=0)[["Resolved", "Pending"]])

# 2. Add total complaints
dept_status["Total"] = dept_status["Resolved"] + dept_status["Pending"]

# 3. Sort & take Top 5 departments
dept_status = dept_status.sort_values("Total", ascending=False)
top5 = dept_status.head(5)

# 4. Combine remaining departments into "Others"
others = pd.DataFrame(top5.iloc[0:0])  # empty structure
others.loc["Others"] = dept_status.iloc[5:][["Resolved", "Pending"]].sum()
others["Total"] = others["Resolved"] + others["Pending"]

final_df = pd.concat([top5, others])

# 5. Shorten department names
dept_short_names = {
    "Municipal Corporation of Delhi (MCD)": "MCD",
    "Public Works Department (PWD)": "PWD",
    "Delhi Development Authority (DDA)": "DDA",
    "Delhi Jal Board (DJB)": "DJB",
    "Irrigation & Flood Control (IFC)": "IFC",
    "Others": "Others"}
final_df.index = final_df.index.to_series().replace(dept_short_names)

# 6. Percentage calculation (NOW CORRECT)
final_df["Resolved %"] = final_df["Resolved"] / final_df["Total"] * 100
final_df["Pending %"] = final_df["Pending"] / final_df["Total"] * 100

# PLOT
departments = final_df.index.tolist()
resolved_pct = final_df["Resolved %"].values
pending_pct = final_df["Pending %"].values

x = np.arange(len(departments))
bar_width = 0.6

fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(x, pending_pct, width=bar_width, color="#e85724", label="Pending")
ax.bar(x, resolved_pct, width=bar_width, color="#8cb73f", bottom=pending_pct, label="Resolved")

# Percentage labels
for i in range(len(x)):
    ax.text(x[i], pending_pct[i] / 2,
            f"{pending_pct[i]:.2f}%",
            ha="center", va="center", fontsize=9, color="white")

    ax.text(x[i], pending_pct[i] + resolved_pct[i] / 2,
            f"{resolved_pct[i]:.2f}%",
            ha="center", va="center", fontsize=9, color="white")

ax.set_xticks(x)
ax.set_xticklabels(departments)
ax.set_ylabel("Percentage of complaints")
ax.set_ylim(0, 100)
ax.set_yticks([0, 20, 40, 60, 80, 100])
ax.set_yticklabels(["0%", "20%", "40%", "60%", "80%", "100%"])

ax.legend(frameon=False)

# TABLE
table_data = [
    final_df["Resolved"].astype(int).tolist(),
    final_df["Pending"].astype(int).tolist()]

plt.table(
    cellText=table_data,
    rowLabels=["Resolved", "Pending"],
    colLabels=departments,
    cellLoc="center",
    loc="bottom",
    bbox=[0.0, -0.38, 1.0, 0.25])

plt.subplots_adjust(bottom=0.32)
plt.show()


***Offence wise Pending Complaint Analysis***

In [ ]:
offence_pending_table = (
    gda_updated_15jan2026
    .groupby("Offences", as_index=False)
    .agg(Pending_Complaints=("Status", lambda x: (x == "Pending").sum()))
    .sort_values("Pending_Complaints", ascending=False))

offence_pending_table
# Top 5 + Others
top5_offence_pending = offence_pending_table.head(5)

others_offence = pd.DataFrame({
    "Offences": ["Others"],
    "Pending_Complaints": [offence_pending_table.iloc[5:]["Pending_Complaints"].sum()]})

final_offence_pending = pd.concat(
    [top5_offence_pending, others_offence], ignore_index=True)

print(offence_pending_table)
print(final_offence_pending)

***Table- Department wise Pending Complaint Analysis***

In [ ]:
# 1. Filter only Pending complaints
pending_df = gda_updated_15jan2026[
    gda_updated_15jan2026["Status"] == "Pending"]

# 2. Department-wise pending count
pending_department_table = (
    pending_df
    .groupby("Department Name", as_index=False)["Compliant ID"]
    .count()
    .rename(columns={"Compliant ID": "Pending Complaint Count"})
    .sort_values("Pending Complaint Count", ascending=False))

# 3. Display table
pending_department_table


***Table- Offence- total complaint- Pending Complaint with %***

In [ ]:
import pandas as pd

# 1. Total complaints per department
total_dept = (
    gda_updated_15jan2026
    .groupby("Department Name")["Compliant ID"]
    .count()
    .reset_index(name="Total Complaints"))

# 2. Pending complaints per department
pending_dept = (
    gda_updated_15jan2026[gda_updated_15jan2026["Status"] == "Pending"]
    .groupby("Department Name")["Compliant ID"]
    .count()
    .reset_index(name="Pending Complaints"))

# 3. Merge both
dept_summary = total_dept.merge(
    pending_dept, on="Department Name", how="left")

# 4. Fill NaN pending values with 0
dept_summary["Pending Complaints"] = dept_summary["Pending Complaints"].fillna(0)

# 5. Calculate pending percentage
dept_summary["Pending Percentage (%)"] = (
    dept_summary["Pending Complaints"] / dept_summary["Total Complaints"] * 100).round(2)

# 6. Sort by pending percentage or count (optional)
dept_summary = dept_summary.sort_values(
    "Pending Percentage (%)", ascending=False)

dept_summary


***Graph- Top Department-Pending complaint count***

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Filter only Pending complaints
pending_df = gda_updated_15jan2026[
    gda_updated_15jan2026["Status"] == "Pending"]

# 2. Department-wise pending complaint count
dept_pending_counts = (
    pending_df
    .groupby("Department Name")["Compliant ID"]
    .count()
    .reset_index(name="Pending Complaint Count")
    .sort_values("Pending Complaint Count", ascending=False))

# 3. Select top 5 departments
top5 = dept_pending_counts.head(5)

# 4. Combine remaining departments into "Others"
others = pd.DataFrame({
    "Department Name": ["Others"],
    "Pending Complaint Count": [
        dept_pending_counts.iloc[5:]["Pending Complaint Count"].sum()]})

final_df = pd.concat([top5, others], ignore_index=True)

# 5. Shorten department names (explicit mapping)
dept_short_names = {
    "Municipal Corporation of Delhi (MCD)": "MCD",
    "Public Works Department (PWD)": "PWD",
    "Delhi Development Authority (DDA)": "DDA",
    "Delhi Jal Board (DJB)": "DJB",
    "National Highway Authority of India (NHAI)": "NHAI",
    "Others": "Others"}

final_df["Department Name"] = final_df["Department Name"].replace(dept_short_names)

# 6. Plot
plt.figure(figsize=(6, 5))
bars = plt.bar(
    final_df["Department Name"],
    final_df["Pending Complaint Count"],
    color="#e85724")   # Pending color (same as earlier plots))

plt.ylabel("Pending Complaint Count")
plt.xlabel("Department")

# 7. Vertical value labels inside bars
for bar in bars:
    h = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        h * 0.95,
        f"{int(h)}",
        ha="center",
        va="top",
        rotation=90,
        fontsize=9,
        color="white")

plt.tight_layout()
plt.show()


***Graph- Monthly Complaints: Pending vs Resolved***

In [ ]:
# Load data
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path)

# Convert Month column (format: Jan 2026) to datetime
df['Month_dt'] = pd.to_datetime(df['Month'], format='%b %Y')

# Keep only Pending and Resolved
df = df[df['Status'].isin(['Pending', 'Resolved'])]

# Filter from Oct 2023 onwards
start_date = pd.to_datetime('Oct 2023', format='%b %Y')
df_filtered = df[df['Month_dt'] >= start_date]

# Group by Month and Status, count unique Complaint IDs
complaint_count = (
    df_filtered.groupby(['Month', 'Status'])['Compliant ID']
    .nunique()
    .unstack(fill_value=0))

# Sort months chronologically
complaint_count = complaint_count.loc[
    df_filtered.drop_duplicates('Month')
    .sort_values('Month_dt')['Month']]

# Plot with custom colors
plt.figure(figsize=(16, 7))
complaint_count.plot(
    kind='bar',
    stacked=True,
    color={
        'Pending': '#e85724',
        'Resolved': '#8cb73f'})

plt.title('Monthly Complaints: Pending vs Resolved')
plt.xlabel('Month-Year')
plt.ylabel('Number of Complaints')
plt.xticks(rotation=90)
plt.legend(title='Status')
plt.tight_layout()
plt.show()


***Dumping-Burning Complaint count***

In [ ]:
import pandas as pd

# Load data
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path)

# Total complaints (ALL offences)
total_complaints = df["Compliant ID"].nunique()

# Dumping & Burning offence list
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Dumping of Construction & Demolition Waste',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste']

# Dumping & Burning complaints (ALL four combined)
df_db = df[df['Offences'].isin(dumping_burning_offences)]

dumping_burning_total = df_db["Compliant ID"].nunique()

dumping_burning_pct = round(
    dumping_burning_total / total_complaints * 100, 2)

# Individual offence-wise calculation
offence_breakdown = (
    df_db.groupby("Offences")["Compliant ID"]
    .nunique()
    .reset_index(name="Complaint Count"))

# Percentage of TOTAL complaints
offence_breakdown["% of Total Complaints"] = (
    offence_breakdown["Complaint Count"] / total_complaints * 100).round(2)

# Output
print(f"Total complaints: {total_complaints}")
print(f"Total Dumping & Burning complaints: {dumping_burning_total}")
print(f"Dumping & Burning % of total complaints: {dumping_burning_pct}%")

offence_breakdown


***Month wise detailed Dumping-Burning Complaint count***

In [ ]:
# Load data
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path)

# Convert Month to datetime
df['Month_dt'] = pd.to_datetime(df['Month'], format='%b %Y')

# TOTAL complaints per month (ALL offences)
monthly_total_complaints = (
    df.groupby('Month_dt')['Compliant ID']
      .nunique()
      .rename("total_complaints"))

# Dumping & Burning complaints
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Dumping of Construction & Demolition Waste',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste']

df_db = df[df['Offences'].isin(dumping_burning_offences)]

monthly_db = (
    df_db.groupby(['Month_dt', 'Offences'])['Compliant ID']
         .nunique()
         .reset_index())

monthly_db_table = (
    monthly_db
    .pivot(index='Month_dt', columns='Offences', values='Compliant ID')
    .fillna(0)
    .astype(int))

# Sum of dumping + burning offences
monthly_db_table["dumping_burning"] = monthly_db_table.sum(axis=1)

# Combine both
final_monthly_table = (
    monthly_db_table
    .merge(monthly_total_complaints, left_index=True, right_index=True, how="left")
    .sort_index())

# Format Month for display
final_monthly_table.index = final_monthly_table.index.strftime('%b %Y')

# Percentage of dumping & burning complaints per month
final_monthly_table["dumping_burning_pct"] = (
    final_monthly_table["dumping_burning"]
    / final_monthly_table["total_complaints"]
    * 100).round(2)

final_monthly_table


***Month wise detailed Dumping-Burning Status wise Complaint count***

In [ ]:
import pandas as pd

# Load data
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path)

# Convert Month to datetime
df['Month_dt'] = pd.to_datetime(df['Month'], format='%b %Y')

# 1. Total complaints (ALL offences)
monthly_total = (
    df.groupby('Month_dt')['Compliant ID']
      .nunique()
      .rename('total_complaints'))

# 2. Dumping & Burning complaints
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Dumping of Construction & Demolition Waste',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste']

df_db = df[df['Offences'].isin(dumping_burning_offences)]

# Total dumping & burning
db_total = (
    df_db.groupby('Month_dt')['Compliant ID']
         .nunique()
         .rename('dumping_burning_total'))

# Dumping & burning by status
db_status = (
    df_db[df_db['Status'].isin(['Pending', 'Resolved'])]
    .groupby(['Month_dt', 'Status'])['Compliant ID']
    .nunique()
    .unstack(fill_value=0)
    .rename(columns={
        'Pending': 'dumping_burning_pending',
        'Resolved': 'dumping_burning_resolved'}))

# 3. Combine everything
final_table = (
    pd.concat([monthly_total, db_total, db_status], axis=1)
      .fillna(0)
      .astype(int)
      .sort_index())

# Format Month for display
final_table.index = final_table.index.strftime('%b %Y')

final_table


***Month wise Comparison (Pending-Resolved)- Dumping-Burning complaint vs Other complaint***

In [ ]:
# Load data
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path)

# Handle Month
df['Month_dt'] = pd.to_datetime(df['Month'], format='%b %Y')

# Filter from Oct 2023 onwards (keep/remove as needed)
start_date = pd.to_datetime('Oct 2023', format='%b %Y')
df = df[df['Month_dt'] >= start_date]

# Define Dumping & Burning offences
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Dumping of Construction & Demolition Waste',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste']

# Create offence group
df['offence_group'] = df['Offences'].isin(dumping_burning_offences)
df['offence_group'] = df['offence_group'].map(
    {True: 'Dumping & Burning', False: 'Other Offences'})

# Keep only Pending & Resolved
df = df[df['Status'].isin(['Pending', 'Resolved'])]

# TOTAL complaints per month
total_monthly = (
    df.groupby(['Month', 'Month_dt'])['Compliant ID']
      .nunique()
      .rename('total_complaints'))

# Complaints by offence group & status
group_status = (
    df.groupby(['Month', 'Month_dt', 'offence_group', 'Status'])['Compliant ID']
      .nunique()
      .unstack(fill_value=0)
      .reset_index())

# Separate Dumping & Burning
db = group_status[group_status['offence_group'] == 'Dumping & Burning']
db = db.set_index(['Month', 'Month_dt'])

db_table = pd.DataFrame({
    'dumping_burning': db['Pending'] + db['Resolved'],
    'dumping_burning_pending': db['Pending'],
    'dumping_burning_resolved': db['Resolved']})

# Separate Other Offences
other = group_status[group_status['offence_group'] == 'Other Offences']
other = other.set_index(['Month', 'Month_dt'])

other_table = pd.DataFrame({
    'other_complaints': other['Pending'] + other['Resolved'],
    'other_complaints_pending': other['Pending'],
    'other_complaints_resolved': other['Resolved']
})

# Combine everything
final_table = (
    pd.concat([total_monthly, db_table, other_table], axis=1)
      .fillna(0)
      .astype(int)
      .sort_index(level='Month_dt')
)

# Format Month for display
final_table.index = final_table.index.get_level_values('Month')

final_table


***Two Scale Graph- Monthly Complaints (Pending-Resolved) vs Dumping & Burning***

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path)

# Handle Month
df['Month_dt'] = pd.to_datetime(df['Month'], format='%b %Y')

# Filter from Oct 2023 onwards
start_date = pd.to_datetime('Oct 2023', format='%b %Y')
df = df[df['Month_dt'] >= start_date]

# PART 1: Pending vs Resolved (ALL complaints)
df_status = df[df['Status'].isin(['Pending', 'Resolved'])]

status_count = (
    df_status.groupby(['Month', 'Status'])['Compliant ID']
    .nunique()
    .unstack(fill_value=0)
)

# PART 2: Dumping & Burning TOTAL complaints
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Dumping of Construction & Demolition Waste',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste'
]

df_db = df[df['Offences'].isin(dumping_burning_offences)]

db_count = (
    df_db.groupby('Month')['Compliant ID']
    .nunique())

# Sort months correctly
month_order = (
    df.drop_duplicates('Month')
      .sort_values('Month_dt')['Month'])

status_count = status_count.loc[month_order]
db_count = db_count.loc[month_order]

# COMBINED PLOT
fig, ax1 = plt.subplots(figsize=(16, 7))

# Stacked bar: Pending vs Resolved
status_count.plot(
    kind='bar',
    stacked=True,
    ax=ax1,
    color={
        'Pending': '#e85724',
        'Resolved': '#8cb73f'})

ax1.set_xlabel('Month-Year')
ax1.set_ylabel('Pending / Resolved Complaints')

# Line plot: Dumping & Burning
ax2 = ax1.twinx()
ax2.plot(
    db_count.index,
    db_count.values,
    marker='o',
    linewidth=2)
ax2.set_ylabel('Dumping & Burning Complaints')

# Titles & formatting
plt.title('Monthly Complaints (Pending-Resolved) vs Dumping & Burning')
ax1.set_xticklabels(month_order, rotation=90)

fig.tight_layout()
plt.show()


***One Scale Graph- Monthly Complaints (Pending-Resolved) vs Dumping & Burning***

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path)

# Handle Month
df['Month_dt'] = pd.to_datetime(df['Month'], format='%b %Y')

# Filter from Oct 2023 onwards
start_date = pd.to_datetime('Oct 2023', format='%b %Y')
df = df[df['Month_dt'] >= start_date]

# PART 1: Pending vs Resolved
df_status = df[df['Status'].isin(['Pending', 'Resolved'])]

status_count = (
    df_status.groupby(['Month', 'Status'])['Compliant ID']
    .nunique()
    .unstack(fill_value=0))

# PART 2: Dumping & Burning
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Dumping of Construction & Demolition Waste',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste']

df_db = df[df['Offences'].isin(dumping_burning_offences)]

db_count = (
    df_db.groupby('Month')['Compliant ID']
    .nunique())

# Correct month order
month_order = (
    df.drop_duplicates('Month')
      .sort_values('Month_dt')['Month'])

status_count = status_count.loc[month_order]
db_count = db_count.loc[month_order]

# PLOT
fig, ax = plt.subplots(figsize=(16, 7))

# Stacked bars: Pending vs Resolved
status_count.plot(
    kind='bar',
    stacked=True,
    ax=ax, color={
        'Pending': '#e85724',
        'Resolved': '#8cb73f'})

# Line plot: Dumping & Burning (SAME AXIS)
ax.plot(
    range(len(db_count)),
    db_count.values,
    marker='o',
    linewidth=2,
    label='Dumping & Burning Complaints')

ax.set_xlabel('Month-Year')
ax.set_ylabel('Number of Complaints')
ax.set_title('Monthly Complaints: Pending–Resolved vs Dumping & Burning')

ax.set_xticklabels(month_order, rotation=90)
ax.legend()

plt.tight_layout()
plt.show()


***Graph- Monthly Complaint Burden by Status (Dumping-Burning vs Others)***

In [ ]:
# Load data
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path)

# Handle Month
df['Month_dt'] = pd.to_datetime(df['Month'], format='%b %Y')

# Filter from Oct 2023 onwards (keep/remove as needed)
start_date = pd.to_datetime('Oct 2023', format='%b %Y')
df = df[df['Month_dt'] >= start_date]

# Define Dumping & Burning offences
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Dumping of Construction & Demolition Waste',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste']

# Create offence group
df['offence_group'] = df['Offences'].isin(dumping_burning_offences)
df['offence_group'] = df['offence_group'].map(
    {True: 'Dumping & Burning', False: 'Other Offences'})

# Keep only Pending & Resolved
df = df[df['Status'].isin(['Pending', 'Resolved'])]

# Aggregate data
monthly_stack = (
    df.groupby(['Month', 'Month_dt', 'offence_group', 'Status'])['Compliant ID']
      .nunique()
      .reset_index())

# Pivot for stacked plotting
pivot = (
    monthly_stack
    .pivot_table(
        index=['Month', 'Month_dt'],
        columns=['offence_group', 'Status'],
        values='Compliant ID',
        fill_value=0).sort_index(level='Month_dt'))

months = pivot.index.get_level_values('Month')

# Plot: Single-scale stacked bar
fig, ax = plt.subplots(figsize=(16, 8))

bottom = None

stack_order = [
    ('Other Offences', 'Resolved'),
    ('Other Offences', 'Pending'),
    ('Dumping & Burning', 'Resolved'),
    ('Dumping & Burning', 'Pending')]

for group, status in stack_order:
    values = pivot[(group, status)]
    ax.bar(
        months,
        values,
        bottom=bottom,
        label=f"{group} – {status}")
    bottom = values if bottom is None else bottom + values

# Formatting
ax.set_xlabel('Month-Year')
ax.set_ylabel('Number of Complaints')
ax.set_title('Monthly Complaint Burden by Status and Offence Type')

plt.xticks(rotation=90)
ax.legend(ncol=2)
plt.tight_layout()
plt.show()


***MSW- Dumping-Burning Complaint count***

In [ ]:
import pandas as pd

# Load data
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path)

# Total complaints (ALL offences)
total_complaints = df["Compliant ID"].nunique()

# Dumping & Burning offence list
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste']

# Dumping & Burning complaints (ALL four combined)
df_db = df[df['Offences'].isin(dumping_burning_offences)]

dumping_burning_total = df_db["Compliant ID"].nunique()

dumping_burning_pct = round(
    dumping_burning_total / total_complaints * 100, 2)

# Individual offence-wise calculation
offence_breakdown = (
    df_db.groupby("Offences")["Compliant ID"]
    .nunique()
    .reset_index(name="Complaint Count"))

# Percentage of TOTAL complaints
offence_breakdown["% of Total Complaints"] = (
    offence_breakdown["Complaint Count"] / total_complaints * 100).round(2)

# Output
print(f"Total complaints: {total_complaints}")
print(f"Total Dumping & Burning complaints: {dumping_burning_total}")
print(f"Dumping & Burning % of total complaints: {dumping_burning_pct}%")

offence_breakdown


***One Scale Graph- Monthly Complaints (Pending-Resolved) vs MSW- Dumping & Burning***

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path)

# Handle Month
df['Month_dt'] = pd.to_datetime(df['Month'], format='%b %Y')

# Filter from Oct 2023 onwards
start_date = pd.to_datetime('Oct 2023', format='%b %Y')
df = df[df['Month_dt'] >= start_date]

# PART 1: Pending vs Resolved
df_status = df[df['Status'].isin(['Pending', 'Resolved'])]

status_count = (
    df_status.groupby(['Month', 'Status'])['Compliant ID']
    .nunique()
    .unstack(fill_value=0))

# PART 2: Dumping & Burning
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste']

df_db = df[df['Offences'].isin(dumping_burning_offences)]

db_count = (
    df_db.groupby('Month')['Compliant ID']
    .nunique())

# Correct month order
month_order = (
    df.drop_duplicates('Month')
      .sort_values('Month_dt')['Month'])

status_count = status_count.loc[month_order]
db_count = db_count.loc[month_order]

# PLOT
fig, ax = plt.subplots(figsize=(16, 7))

# Stacked bars: Pending vs Resolved
status_count.plot(
    kind='bar',
    stacked=True,
    ax=ax, color={
        'Pending': '#e85724',
        'Resolved': '#8cb73f'})

# Line plot: Dumping & Burning (SAME AXIS)
ax.plot(
    range(len(db_count)),
    db_count.values,
    marker='o',
    linewidth=2,
    label='Dumping & Burning Complaints')

ax.set_xlabel('Month-Year')
ax.set_ylabel('Number of Complaints')
ax.set_title('Monthly Complaints: Pending–Resolved vs Dumping & Burning')

ax.set_xticklabels(month_order, rotation=90)
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. LOAD DATA
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path, encoding="latin1")

# 2. DEFINE DUMPING & BURNING OFFENCES (EXACT MATCH)
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste']

# 3. COUNT COMPLAINTS
dump_burn_df = df[df['Offences'].isin(dumping_burning_offences)]

dump_burn_count = len(dump_burn_df)
total_count = len(df)
other_count = total_count - dump_burn_count

# Percentage
dump_burn_pct = round((dump_burn_count / total_count) * 100, 1)
other_pct = round(100 - dump_burn_pct, 1)

# 4. DATA FOR DONUT CHART
sizes = [dump_burn_count, other_count]
labels = [
    f"Dumping & Burning\n{dump_burn_pct}%",
    f"Other Complaints\n{other_pct}%"]

# 5. PLOT DONUT CHART
plt.figure(figsize=(6, 6))
plt.pie(
    sizes,
    labels=labels,
    startangle=90,
    wedgeprops=dict(width=0.4))

plt.title(
    "Share of Citizen Complaints by Offence Type",
    fontsize=14)

plt.tight_layout()
plt.show()

# 6. PRINT COUNTS (FOR REPORTING / QA)
print("Total complaints:", total_count)
print("Dumping & burning complaints:", dump_burn_count)
print("Other complaints:", other_count)
print("Dumping & burning share (%):", dump_burn_pct)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. LOAD DATA
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path, encoding="latin1")

# 2. FILTER: DUMPING & BURNING OFFENCES ONLY
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste'
]

df_db = df[df['Offences'].isin(dumping_burning_offences)].copy()

# 3. COUNT REPEATED COMPLAINTS BY LOCATION
location_counts = (
    df_db
    .groupby('Geo Location')['Compliant ID']
    .nunique()
    .reset_index(name='Complaint_Count')
)

# 4. SELECT TOP 10 LOCATIONS
top10_locations = (
    location_counts
    .sort_values(by='Complaint_Count', ascending=False)
    .head(10)
    .sort_values(by='Complaint_Count')  # for clean plotting
)

# 5. PLOT BAR CHART
plt.figure(figsize=(8, 5))

plt.barh(
    top10_locations['Geo Location'],
    top10_locations['Complaint_Count']
)

plt.xlabel("Number of Complaints")
plt.ylabel("Location")
plt.title(
    "Top 10 Locations with Repeated Dumping & Burning Complaints",
    fontsize=13)

plt.tight_layout()
plt.show()

# 6. PRINT TABLE (FOR REPORTING / QA)
print("\nTop 10 Repeated Dumping & Burning Locations:")
print(top10_locations.sort_values(by='Complaint_Count', ascending=False))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. LOAD DATA
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path, encoding="latin1")

# 2. FILTER: DUMPING & BURNING OFFENCES ONLY
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste'
]

df_db = df[df['Offences'].isin(dumping_burning_offences)].copy()

# 3. COUNT COMPLAINTS PER LOCATION
location_counts = (
    df_db
    .groupby('Geo Location')['Compliant ID']
    .nunique()
    .reset_index(name='Complaint_Count'))

# 4. TOP 10 REPEATED LOCATIONS

top10 = (
    location_counts
    .sort_values(by='Complaint_Count', ascending=False)
    .head(10)
    .sort_values(by='Complaint_Count')  # sort for lollipop)

# 5. LOLLIPOP CHART
plt.figure(figsize=(8, 5))

# Draw lines
plt.hlines(
    y=top10['Geo Location'],
    xmin=0,
    xmax=top10['Complaint_Count'],
    linewidth=2)

# Draw dots
plt.plot(
    top10['Complaint_Count'],
    top10['Geo Location'],
    "o")

plt.xlabel("Number of Complaints")
plt.ylabel("Location")
plt.title(
    "Top 10 Locations with Repeated Dumping & Burning Complaints",
    fontsize=13)

plt.tight_layout()
plt.show()

# 6. PRINT TABLE FOR REFERENCE
print("\nTop 10 Repeated Dumping & Burning Locations:")
print(top10.sort_values(by='Complaint_Count', ascending=False))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. LOAD DATA
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path, encoding="latin1")

# 2. FILTER DUMPING & BURNING OFFENCES
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste']

df = df[df['Offences'].isin(dumping_burning_offences)].copy()

# 3. COMPLAINT COUNT PER GEO LOCATION
location_counts = (
    df
    .groupby('Geo Location')
    .size()
    .reset_index(name='Complaint_Count'))

total_complaints = location_counts['Complaint_Count'].sum()

# 4. CALCULATE PERCENTAGE SHARE PER LOCATION
location_counts['Percentage'] = (
    location_counts['Complaint_Count'] / total_complaints
) * 100

# 5. CLASSIFY DUMP INTENSITY
def classify_waste_dump(percentage):
    if percentage < 4:
        return "Low Dump"
    elif percentage <= 35:
        return "Medium Dump"
    else:
        return "Heavy Dump"

location_counts['Dump_Intensity'] = (
    location_counts['Percentage']
    .apply(classify_waste_dump)
)

# 6. INTENSITY DISTRIBUTION (FOR PLOTTING)
intensity_counts = (
    location_counts['Dump_Intensity']
    .value_counts()
    .reindex(['Heavy Dump', 'Medium Dump', 'Low Dump'])
    .fillna(0)
)

# 7. STACKED BAR PLOT (BAQ-READY)
fig, ax = plt.subplots(figsize=(6, 5))

bottom = 0
for intensity in ['Heavy Dump', 'Medium Dump', 'Low Dump']:
    ax.bar(
        ['Dumping & Burning Sites'],
        intensity_counts[intensity],
        bottom=bottom,
        label=intensity
    )
    bottom += intensity_counts[intensity]

ax.set_ylabel("Number of Locations", fontsize=11)
ax.set_title(
    "High-Intensity Dumping Sites Are Fewer but More Damaging",
    fontsize=13
)

ax.legend(title="Dump Intensity", frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------
# 1. LOAD DATA
# --------------------------------------------------
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path, encoding="latin1")

# --------------------------------------------------
# 2. FILTER DUMPING & BURNING OFFENCES
# --------------------------------------------------
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste'
]

df = df[df['Offences'].isin(dumping_burning_offences)].copy()

# --------------------------------------------------
# 3. COUNT COMPLAINTS PER GEO LOCATION
# --------------------------------------------------
location_counts = (
    df
    .groupby('Geo Location')
    .size()
    .reset_index(name='Complaint_Count')
)

# --------------------------------------------------
# 4. CLASSIFY DUMP INTENSITY (CORRECT LOGIC)
# --------------------------------------------------
def classify_waste_dump(count):
    if count <= 2:
        return "Low Dump"
    elif count <= 5:
        return "Medium Dump"
    else:
        return "Heavy Dump"

location_counts['Dump_Intensity'] = (
    location_counts['Complaint_Count']
    .apply(classify_waste_dump)
)

# --------------------------------------------------
# 5. COUNT LOCATIONS BY INTENSITY
# --------------------------------------------------
intensity_summary = (
    location_counts['Dump_Intensity']
    .value_counts()
    .reindex(['Heavy Dump', 'Medium Dump', 'Low Dump'])
    .fillna(0)
)

# --------------------------------------------------
# 6. BAQ-READY STACKED BAR CHART
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 5))

bottom = 0
for intensity in ['Heavy Dump', 'Medium Dump', 'Low Dump']:
    ax.bar(
        ['Dumping & Burning Locations'],
        intensity_summary[intensity],
        bottom=bottom,
        label=intensity
    )
    bottom += intensity_summary[intensity]

ax.set_ylabel("Number of Locations", fontsize=11)
ax.set_title(
    "High-Intensity Dumping Sites Are Fewer but More Damaging",
    fontsize=13
)

ax.legend(title="Dump Intensity", frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------
# 1. LOAD DATA
# --------------------------------------------------
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path, encoding="latin1")

# --------------------------------------------------
# 2. DEFINE DUMPING & BURNING OFFENCES
# --------------------------------------------------
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste'
]

# --------------------------------------------------
# 3. FILTER VALID STATUS ONLY
# --------------------------------------------------
valid_status = ['Resolved', 'Pending']
df = df[df['Status'].isin(valid_status)].copy()

# --------------------------------------------------
# 4. SPLIT DATASETS
# --------------------------------------------------
dump_burn_df = df[df['Offences'].isin(dumping_burning_offences)]
other_df = df[~df['Offences'].isin(dumping_burning_offences)]

# --------------------------------------------------
# 5. STATUS COUNTS
# --------------------------------------------------
dump_burn_status = dump_burn_df['Status'].value_counts().reindex(valid_status, fill_value=0)
other_status = other_df['Status'].value_counts().reindex(valid_status, fill_value=0)

# --------------------------------------------------
# 6. BAR CHART
# --------------------------------------------------
labels = valid_status
x = range(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 5))

ax.bar(
    [i - width/2 for i in x],
    dump_burn_status.values,
    width,
    label='Dumping & Burning'
)

ax.bar(
    [i + width/2 for i in x],
    other_status.values,
    width,
    label='Other Complaints'
)

# --------------------------------------------------
# 7. FORMATTING
# --------------------------------------------------
ax.set_ylabel('Number of Complaints', fontsize=11)
ax.set_title(
    'Resolution Gap in Dumping & Burning Complaints',
    fontsize=13
)

ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.legend(frameon=False)

plt.tight_layout()
plt.show()


***Map- Delhi Dumping Burning Complaints***


In [ ]:
# Imports
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import Point, box
import matplotlib.pyplot as plt
import matplotlib.colors as colors

# Load CSV
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
complaints_df = pd.read_csv(csv_path, encoding="latin1")

# Fix mojibake text
def fix_mojibake(x):
    if isinstance(x, str):
        try:
            return x.encode("latin1").decode("utf-8")
        except:
            return x
    return x

complaints_df = complaints_df.applymap(fix_mojibake)

# FILTER: Dumping & Burning offences ONLY
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Dumping of Construction & Demolition Waste',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste']

complaints_df = complaints_df[complaints_df['Offences'].isin(dumping_burning_offences)]

# Split Latitude / Longitude
complaints_df[['Latitude', 'Longitude']] = (
    complaints_df['Latitude & Longitude'].str.split(',', expand=True))

complaints_df['Latitude'] = complaints_df['Latitude'].astype(float)
complaints_df['Longitude'] = complaints_df['Longitude'].astype(float)

# Load Delhi Ward Shapefile
shp_path = r'C:\Users\Jeen priyee\Downloads\Delhi_Wards-SHP_Datameet\Delhi_Wards.shp'
delhi_gdf = gpd.read_file(shp_path)

if delhi_gdf.crs is None:
    delhi_gdf.set_crs(epsg=4326, inplace=True)

# Create GeoDataFrame for complaints
geometry = [
    Point(xy) for xy in zip(
        complaints_df['Longitude'],
        complaints_df['Latitude'])]

complaints_gdf = gpd.GeoDataFrame(
    complaints_df,
    geometry=geometry,
    crs=delhi_gdf.crs)

# Spatial Join (point → ward)
joined_gdf = gpd.sjoin(
    complaints_gdf,
    delhi_gdf,
    how='left',
    predicate='within')

# Count complaints per ward
region_counts = joined_gdf.groupby('index_right').size()

delhi_gdf['occurrences'] = region_counts.reindex(
    delhi_gdf.index,
    fill_value=0)

# Identify Ward Name Column
ward_name_col = None
for col in ['name', 'NAME', 'Ward_Name', 'WARD_NAME']:
    if col in delhi_gdf.columns:
        ward_name_col = col
        break

ward_info = delhi_gdf[[ward_name_col]].copy() if ward_name_col else delhi_gdf.copy()
ward_info['occurrences'] = delhi_gdf['occurrences']

top_5_wards = ward_info.sort_values(
    'occurrences',
    ascending=False).head(5)

print("\nTop 5 Wards (Dumping & Burning complaints):")
print(top_5_wards)

# FOLIUM MAP (HTML)
m = folium.Map(location=[28.7041, 77.1025], zoom_start=11)

folium.Choropleth(
    geo_data=delhi_gdf.__geo_interface__,
    data=delhi_gdf,
    columns=[delhi_gdf.index, 'occurrences'],
    key_on='feature.id',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Dumping & Burning Complaints'
).add_to(m)

m.save('delhi_dumping_burning_complaints_map.html')

# STATIC MAP (JPEG)
fig, ax = plt.subplots(figsize=(20, 6), facecolor='white')
ax.set_facecolor('white')

colors_list = [
    '#ffffcc', '#ffeda0', '#fed976', '#feb24c',
    '#fd8d3c', '#fc4e2a', '#e31a1c', '#bd0026', '#800026']
cmap = colors.LinearSegmentedColormap.from_list('YlOrRd', colors_list)

delhi_gdf.plot(
    column='occurrences',
    cmap=cmap,
    linewidth=0.8,
    edgecolor='black',
    ax=ax,
    alpha=0.75)

# Highlight Top 5 wards
for idx in top_5_wards.index:
    delhi_gdf.loc[[idx]].boundary.plot(
        ax=ax,
        color='black',
        linewidth=2.5)

ax.set_axis_off()
plt.title('Delhi: Dumping & Burning Complaints by Ward', fontsize=20)

sm = plt.cm.ScalarMappable(
    cmap=cmap,
    norm=plt.Normalize(
        vmin=delhi_gdf['occurrences'].min(),
        vmax=delhi_gdf['occurrences'].max()))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, orientation='horizontal', pad=0.04)
cbar.set_label('Number of Dumping & Burning Complaints', fontsize=14)

plt.savefig(
    'delhi_dumping_burning_complaints_map.jpg',
    dpi=300,
    bbox_inches='tight',
    facecolor='white')

plt.show()


***Map- Delhi MSW Dumping Burning Complaints***


In [ ]:
# Imports
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import Point, box
import matplotlib.pyplot as plt
import matplotlib.colors as colors

# Load CSV
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
complaints_df = pd.read_csv(csv_path, encoding="latin1")

# Fix mojibake text
def fix_mojibake(x):
    if isinstance(x, str):
        try:
            return x.encode("latin1").decode("utf-8")
        except:
            return x
    return x

complaints_df = complaints_df.applymap(fix_mojibake)

# FILTER: Dumping & Burning offences ONLY
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste']

complaints_df = complaints_df[complaints_df['Offences'].isin(dumping_burning_offences)]

# Split Latitude / Longitude
complaints_df[['Latitude', 'Longitude']] = (
    complaints_df['Latitude & Longitude'].str.split(',', expand=True))

complaints_df['Latitude'] = complaints_df['Latitude'].astype(float)
complaints_df['Longitude'] = complaints_df['Longitude'].astype(float)

# Load Delhi Ward Shapefile
shp_path = r'C:\Users\Jeen priyee\Downloads\Delhi_Wards-SHP_Datameet\Delhi_Wards.shp'
delhi_gdf = gpd.read_file(shp_path)

if delhi_gdf.crs is None:
    delhi_gdf.set_crs(epsg=4326, inplace=True)

# Create GeoDataFrame for complaints
geometry = [
    Point(xy) for xy in zip(
        complaints_df['Longitude'],
        complaints_df['Latitude'])]

complaints_gdf = gpd.GeoDataFrame(
    complaints_df,
    geometry=geometry,
    crs=delhi_gdf.crs)

# Spatial Join (point → ward)
joined_gdf = gpd.sjoin(
    complaints_gdf,
    delhi_gdf,
    how='left',
    predicate='within')

# Count complaints per ward
region_counts = joined_gdf.groupby('index_right').size()

delhi_gdf['occurrences'] = region_counts.reindex(
    delhi_gdf.index,
    fill_value=0)

# Identify Ward Name Column
ward_name_col = None
for col in ['name', 'NAME', 'Ward_Name', 'WARD_NAME']:
    if col in delhi_gdf.columns:
        ward_name_col = col
        break

ward_info = delhi_gdf[[ward_name_col]].copy() if ward_name_col else delhi_gdf.copy()
ward_info['occurrences'] = delhi_gdf['occurrences']

top_5_wards = ward_info.sort_values(
    'occurrences',
    ascending=False).head(5)

print("\nTop 5 Wards (Dumping & Burning complaints):")
print(top_5_wards)

# FOLIUM MAP (HTML)
m = folium.Map(location=[28.7041, 77.1025], zoom_start=11)

folium.Choropleth(
    geo_data=delhi_gdf.__geo_interface__,
    data=delhi_gdf,
    columns=[delhi_gdf.index, 'occurrences'],
    key_on='feature.id',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Dumping & Burning Complaints'
).add_to(m)

m.save('delhi_MSW_dumping_burning_complaints_map.html')

# STATIC MAP (JPEG)
fig, ax = plt.subplots(figsize=(20, 6), facecolor='white')
ax.set_facecolor('white')

colors_list = [
    '#ffffcc', '#ffeda0', '#fed976', '#feb24c',
    '#fd8d3c', '#fc4e2a', '#e31a1c', '#bd0026', '#800026']
cmap = colors.LinearSegmentedColormap.from_list('YlOrRd', colors_list)

delhi_gdf.plot(
    column='occurrences',
    cmap=cmap,
    linewidth=0.8,
    edgecolor='black',
    ax=ax,
    alpha=0.75)

# Highlight Top 5 wards
for idx in top_5_wards.index:
    delhi_gdf.loc[[idx]].boundary.plot(
        ax=ax,
        color='black',
        linewidth=2.5)

ax.set_axis_off()
plt.title('Delhi: MSW Dumping & Burning Complaints by Ward', fontsize=20)

sm = plt.cm.ScalarMappable(
    cmap=cmap,
    norm=plt.Normalize(
        vmin=delhi_gdf['occurrences'].min(),
        vmax=delhi_gdf['occurrences'].max()))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, orientation='horizontal', pad=0.04)
cbar.set_label('Number of Dumping & Burning Complaints', fontsize=14)

plt.savefig(
    'delhi_msw_dumping_burning_complaints_map.jpg',
    dpi=300,
    bbox_inches='tight',
    facecolor='white')

plt.show()


***MSW Dumping-Burning Citywide hotspots_diagnostic***

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
import os

# 0. LOAD DATA
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path, encoding="latin1")

# 1. FILTER: DUMPING & BURNING OFFENCES
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste'
]

df = df[df['Offences'].isin(dumping_burning_offences)].copy()

# 2. EXTRACT & CLEAN LAT/LON
df[['Latitude', 'Longitude']] = df['Latitude & Longitude'].str.split(',', expand=True)
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# Flag missing or zero coordinates
df['missing_latlon'] = df['Latitude'].isna() | df['Longitude'].isna()
df['zero_coordinates'] = (df['Latitude'] == 0) & (df['Longitude'] == 0)

# Keep only valid coordinates
df_valid = df[~df['missing_latlon'] & ~df['zero_coordinates']].copy()

# 3. DBSCAN CLUSTERING (5 METERS)
df_valid['lat_rad'] = np.radians(df_valid['Latitude'])
df_valid['lon_rad'] = np.radians(df_valid['Longitude'])
coords = df_valid[['lat_rad', 'lon_rad']].values

EARTH_RADIUS_M = 6_371_000
eps = 5 / EARTH_RADIUS_M  # 5 meters

db = DBSCAN(eps=eps, min_samples=1, metric='haversine').fit(coords)
df_valid['Cluster'] = db.labels_

# 4. HOTSPOT DIAGNOSTIC
cluster_counts = (
    df_valid.groupby('Cluster')['Compliant ID']
    .nunique()
    .reset_index(name='Count')
)

single_complaints = cluster_counts.loc[cluster_counts['Count'] == 1, 'Count'].sum()
recurring_complaints = cluster_counts.loc[cluster_counts['Count'] >= 2, 'Count'].sum()
total_complaints = len(df_valid)

diagnostic_summary = pd.DataFrame({
    'Total dumping & burning complaints': [total_complaints],
    'Single-occurrence complaints': [single_complaints],
    'Recurring hotspot complaints (≥2)': [recurring_complaints],
    'Number of recurring hotspots': [len(cluster_counts[cluster_counts['Count'] >= 2])]
})

print("===== HOTSPOT DIAGNOSTIC =====")
print(diagnostic_summary)

# 5. KEEP ONLY RECURRING HOTSPOTS (≥ 2 complaints)
recurring_clusters = cluster_counts.loc[cluster_counts['Count'] >= 2, 'Cluster']
df_recurring = df_valid[df_valid['Cluster'].isin(recurring_clusters)].copy()

# 6. LOCATION-LEVEL SUMMARY
location_summary = (
    df_recurring.groupby('Cluster')
    .agg(
        Geo_Location=('Geo Location', 'first'),
        Count=('Compliant ID', 'nunique'),
        Latitude=('Latitude', 'mean'),
        Longitude=('Longitude', 'mean')).reset_index())

# 7. OFFENCE TYPES WITH COUNTS
offence_counts = (
    df_recurring.groupby(['Cluster', 'Offences'])['Compliant ID']
    .nunique()
    .reset_index(name='cnt'))

offence_text = (
    offence_counts
    .assign(offence_str=lambda x: x['Offences'] + ' (' + x['cnt'].astype(str) + ')')
    .groupby('Cluster')['offence_str']
    .apply(', '.join)
    .reset_index(name='Offences'))

# 8. FINAL HOTSPOT TABLE
final_table = (
    location_summary.merge(offence_text, on='Cluster', how='left')
    [['Geo_Location', 'Count', 'Offences', 'Latitude', 'Longitude']])

# Sort and label hotspots
final_table_sorted = final_table.sort_values(by='Count', ascending=False).reset_index(drop=True)
final_table_sorted.insert(0, 'Hotspot', ['Hotspot ' + str(i + 1) for i in range(len(final_table_sorted))])

# 9. EXPORT
#download_path = os.path.join(os.path.expanduser("~"), "Downloads", "msw_dumping_burning_citywide_hotspots_diagnostic.xlsx")
#final_table_sorted.to_excel(download_path, index=False)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_rows", None)

print("\n===== FIELD-READY HOTSPOTS =====")
print(final_table_sorted)
#print("\nExcel saved at:", download_path)


***MSW Dumping-Burning Citywide Hotspot with Ward***

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
import geopandas as gpd
from shapely.geometry import Point
import os

# 0. LOAD DATA
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path, encoding="latin1")

# 1. FILTER: DUMPING & BURNING OFFENCES ONLY
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste']

df = df[df['Offences'].isin(dumping_burning_offences)].copy()

# 2. EXTRACT & CLEAN LATITUDE / LONGITUDE
df[['Latitude', 'Longitude']] = (
    df['Latitude & Longitude']
    .astype(str)
    .str.split(',', expand=True))

df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

df = df.dropna(subset=['Latitude', 'Longitude'])

# 3. DBSCAN CLUSTERING (5 METERS)
df['lat_rad'] = np.radians(df['Latitude'])
df['lon_rad'] = np.radians(df['Longitude'])

coords = df[['lat_rad', 'lon_rad']].values

EARTH_RADIUS_M = 6_371_000
eps = 5 / EARTH_RADIUS_M  # 5 meters

db = DBSCAN(
    eps=eps,
    min_samples=1,
    metric='haversine').fit(coords)

df['Cluster'] = db.labels_

# 4. KEEP ONLY RECURRING LOCATIONS (≥ 2 COMPLAINTS)
cluster_counts = (
    df.groupby('Cluster')['Compliant ID']
    .nunique()
    .reset_index(name='Count'))

recurring_clusters = cluster_counts.loc[
    cluster_counts['Count'] >= 2, 'Cluster']

df_recurring = df[df['Cluster'].isin(recurring_clusters)].copy()

# 5. LOCATION-LEVEL SUMMARY
location_summary = (
    df_recurring
    .groupby('Cluster')
    .agg(
        Geo_Location=('Geo Location', 'first'),
        Count=('Compliant ID', 'nunique'),
        Latitude=('Latitude', 'mean'),
        Longitude=('Longitude', 'mean'))
    .reset_index())

# 6. OFFENCE TYPES WITH COUNTS
offence_counts = (
    df_recurring
    .groupby(['Cluster', 'Offences'])['Compliant ID']
    .nunique()
    .reset_index(name='cnt'))

offence_text = (
    offence_counts
    .assign(
        offence_str=lambda x:
        x['Offences'] + ' (' + x['cnt'].astype(str) + ')')
    .groupby('Cluster')['offence_str']
    .apply(', '.join)
    .reset_index(name='Offences'))

# 7. MERGE HOTSPOT TABLE
final_table = (
    location_summary
    .merge(offence_text, on='Cluster', how='left')
    [['Cluster', 'Geo_Location', 'Count', 'Offences', 'Latitude', 'Longitude']])

# 8. SPATIAL JOIN: ADD WARD NAME

# Convert hotspots to GeoDataFrame
geometry = [Point(xy) for xy in zip(final_table['Longitude'], final_table['Latitude'])]

hotspot_gdf = gpd.GeoDataFrame(
    final_table,
    geometry=geometry,
    crs="EPSG:4326")

# Load Delhi Ward Shapefile
shp_path = r"C:\Users\Jeen priyee\Downloads\Delhi_Wards-SHP_Datameet\Delhi_Wards.shp"
delhi_gdf = gpd.read_file(shp_path)

# Ensure CRS match
delhi_gdf = delhi_gdf.to_crs("EPSG:4326")

# Spatial join
hotspot_with_ward = gpd.sjoin(
    hotspot_gdf,
    delhi_gdf,
    how="left",
    predicate="within")

# 🔴 CHANGE THIS COLUMN NAME IF NEEDED
WARD_COL = "WARD_NAME"   # <-- inspect shapefile if this errors

hotspot_with_ward = hotspot_with_ward.rename(
    columns={WARD_COL: "Ward Name"})

# 9. FINAL CLEAN TABLE
final_output = (
    hotspot_with_ward
    .drop(columns=['geometry', 'index_right'])
    .sort_values(by='Count', ascending=False)
    .reset_index(drop=True))

final_output.insert(
    0,
    'Hotspot',
    ['Hotspot ' + str(i + 1) for i in range(len(final_output))])

# 10. EXPORT
#download_path = os.path.join(os.path.expanduser("~"), "Downloads", "msw_dumping_burning_citywide_hotspots_with_ward.xlsx")

#final_output.to_excel(download_path, index=False)

# DISPLAY
pd.set_option("display.max_rows", None)
print(final_output)

#print("\nExcel saved at:")
#print(download_path)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
import geopandas as gpd
from shapely.geometry import Point

# 0. LOAD DATA (SAFE)
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path, encoding="latin1")

# Mandatory columns check
required_cols = [
    "Compliant ID", "Offences", "Latitude & Longitude", "Geo Location"
]
missing = set(required_cols) - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# 1. FILTER OFFENCES (STRICT)
dumping_burning_offences = {
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste'
}

df = df[df["Offences"].isin(dumping_burning_offences)].copy()

# 2. EXTRACT & CLEAN COORDINATES (FULL SAFETY)
coords_split = (
    df["Latitude & Longitude"]
    .astype(str)
    .str.replace(" ", "")
    .str.split(",", expand=True))

df["Latitude"] = pd.to_numeric(coords_split[0], errors="coerce")
df["Longitude"] = pd.to_numeric(coords_split[1], errors="coerce")

# Remove invalid coordinates
df = df.dropna(subset=["Latitude", "Longitude"])

# Remove zero & out-of-range coordinates
df = df[
    (df["Latitude"].between(-90, 90)) &
    (df["Longitude"].between(-180, 180)) &
    ~((df["Latitude"] == 0) & (df["Longitude"] == 0))
].copy()

# 3. DBSCAN CLUSTERING (SAFE SPATIAL)
df["lat_rad"] = np.radians(df["Latitude"])
df["lon_rad"] = np.radians(df["Longitude"])

coords = df[["lat_rad", "lon_rad"]].values

EARTH_RADIUS_M = 6_371_000
EPS_METERS = 10   # safer than 5m for GPS noise
eps = EPS_METERS / EARTH_RADIUS_M

db = DBSCAN(
    eps=eps,
    min_samples=1,
    metric="haversine"
)

df["Cluster"] = db.fit_predict(coords)

# 4. KEEP ONLY RECURRING LOCATIONS
cluster_counts = (
    df.groupby("Cluster")["Compliant ID"]
    .nunique()
    .reset_index(name="Count"))

recurring_clusters = cluster_counts.loc[
    cluster_counts["Count"] >= 2, "Cluster"
]

df = df[df["Cluster"].isin(recurring_clusters)].copy()

# 5. LOCATION SUMMARY (ROBUST)
location_summary = (
    df.groupby("Cluster")
    .agg(
        Geo_Location=("Geo Location", "first"),
        Count=("Compliant ID", "nunique"),
        Latitude=("Latitude", "mean"),
        Longitude=("Longitude", "mean")).reset_index())

# 6. OFFENCE COMPOSITION
offence_counts = (
    df.groupby(["Cluster", "Offences"])["Compliant ID"]
    .nunique()
    .reset_index(name="cnt"))

offence_text = (
    offence_counts
    .assign(
        offence_str=lambda x:
        x["Offences"] + " (" + x["cnt"].astype(str) + ")"
    )
    .groupby("Cluster")["offence_str"]
    .apply(", ".join)
    .reset_index(name="Offences"))

final_table = (
    location_summary
    .merge(offence_text, on="Cluster", how="left"))

# 7. SPATIAL JOIN WITH WARDS (BOUNDARY-SAFE)
hotspot_gdf = gpd.GeoDataFrame(
    final_table,
    geometry=[
        Point(xy)
        for xy in zip(final_table["Longitude"], final_table["Latitude"])
    ], crs="EPSG:4326")

shp_path = r"C:\Users\Jeen priyee\Downloads\Delhi_Wards-SHP_Datameet\Delhi_Wards.shp"
delhi_gdf = gpd.read_file(shp_path).to_crs("EPSG:4326")

# Auto-detect ward name column
ward_col = next(
    (c for c in delhi_gdf.columns if "ward" in c.lower()),
    None)

if ward_col is None:
    raise ValueError("Ward name column not found in shapefile")

hotspot_with_ward = gpd.sjoin(
    hotspot_gdf,
    delhi_gdf,
    how="left",
    predicate="intersects")   # safer than "within"

hotspot_with_ward = hotspot_with_ward.rename(
    columns={ward_col: "Ward Name"})

# 8. FINAL CLEAN OUTPUT
final_output = (
    hotspot_with_ward
    .drop(columns=["geometry", "index_right"], errors="ignore")
    .sort_values("Count", ascending=False)
    .reset_index(drop=True))

final_output.insert(0, "Hotspot", [f"Hotspot {i+1}" for i in range(len(final_output))])

pd.set_option("display.max_rows", None)
print(final_output)


***MSW Dumping-Burning Hotspots Jan2025_onwards***

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
import os

# 0. LOAD DATA
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path, encoding="latin1")

# 0a. CONVERT MONTH TO DATETIME
# Assuming the 'Month' column is in 'Jan 2025' format
df['Month_dt'] = pd.to_datetime(df['Month'], format='%b %Y', errors='coerce')

# Filter for Jan 2025 till now
start_date = pd.to_datetime('Jan 2025', format='%b %Y')
df = df[df['Month_dt'] >= start_date].copy()

print(f"Records after date filter: {len(df)}")

# 1. FILTER: DUMPING & BURNING OFFENCES
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste']

df = df[df['Offences'].isin(dumping_burning_offences)].copy()
print(f"Records after offence filter: {len(df)}")

# 2. EXTRACT & CLEAN LAT/LON
df[['Latitude', 'Longitude']] = df['Latitude & Longitude'].str.split(',', expand=True)
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# Flag missing or zero coordinates
df['missing_latlon'] = df['Latitude'].isna() | df['Longitude'].isna()
df['zero_coordinates'] = (df['Latitude'] == 0) & (df['Longitude'] == 0)

# Keep only valid coordinates
df_valid = df[~df['missing_latlon'] & ~df['zero_coordinates']].copy()
print(f"Records with valid coordinates: {len(df_valid)}")

# 3. DBSCAN CLUSTERING (5 METERS)
df_valid['lat_rad'] = np.radians(df_valid['Latitude'])
df_valid['lon_rad'] = np.radians(df_valid['Longitude'])
coords = df_valid[['lat_rad', 'lon_rad']].values

EARTH_RADIUS_M = 6_371_000
eps = 5 / EARTH_RADIUS_M  # 5 meters

db = DBSCAN(eps=eps, min_samples=1, metric='haversine').fit(coords)
df_valid['Cluster'] = db.labels_

# 4. HOTSPOT DIAGNOSTIC
cluster_counts = (
    df_valid.groupby('Cluster')['Compliant ID']
    .nunique()
    .reset_index(name='Count'))

single_complaints = cluster_counts.loc[cluster_counts['Count'] == 1, 'Count'].sum()
recurring_complaints = cluster_counts.loc[cluster_counts['Count'] >= 2, 'Count'].sum()
total_complaints = len(df_valid)

diagnostic_summary = pd.DataFrame({
    'Total dumping & burning complaints': [total_complaints],
    'Single-occurrence complaints': [single_complaints],
    'Recurring hotspot complaints (≥2)': [recurring_complaints],
    'Number of recurring hotspots': [len(cluster_counts[cluster_counts['Count'] >= 2])]})

print("\n===== HOTSPOT DIAGNOSTIC =====")
print(diagnostic_summary)

# 5. KEEP ONLY RECURRING HOTSPOTS (≥ 2 complaints)
recurring_clusters = cluster_counts.loc[cluster_counts['Count'] >= 2, 'Cluster']
df_recurring = df_valid[df_valid['Cluster'].isin(recurring_clusters)].copy()

# 6. LOCATION-LEVEL SUMMARY
location_summary = (
    df_recurring.groupby('Cluster')
    .agg(
        Geo_Location=('Geo Location', 'first'),
        Count=('Compliant ID', 'nunique'),
        Latitude=('Latitude', 'mean'),
        Longitude=('Longitude', 'mean')).reset_index())

# 7. OFFENCE TYPES WITH COUNTS
offence_counts = (
    df_recurring.groupby(['Cluster', 'Offences'])['Compliant ID']
    .nunique()
    .reset_index(name='cnt'))

offence_text = (
    offence_counts
    .assign(offence_str=lambda x: x['Offences'] + ' (' + x['cnt'].astype(str) + ')')
    .groupby('Cluster')['offence_str']
    .apply(', '.join)
    .reset_index(name='Offences'))

# 8. FINAL HOTSPOT TABLE
final_table = (
    location_summary.merge(offence_text, on='Cluster', how='left')
    [['Geo_Location', 'Count', 'Offences', 'Latitude', 'Longitude']])

# Sort and label hotspots
final_table_sorted = final_table.sort_values(by='Count', ascending=False).reset_index(drop=True)
final_table_sorted.insert(0, 'Hotspot', ['Hotspot ' + str(i + 1) for i in range(len(final_table_sorted))])

# 9. EXPORT
#download_path = os.path.join(os.path.expanduser("~"), "Downloads", "msw_dumping_burning_hotspots_Jan2025_onwards.xlsx")
#final_table_sorted.to_excel(download_path, index=False)

pd.set_option("display.max_rows", None)
print("\n===== FIELD-READY HOTSPOTS =====")
print(final_table_sorted)
#print("\nExcel saved at:", download_path)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
import os

# 0. LOAD DATA
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path, encoding="latin1")

# 1. MONTH CLEANING & FILTERING (Jan 2025 onwards)
# Convert Month to datetime (known format: 'Jan 2025')
df["Month_dt"] = pd.to_datetime(
    df["Month"],
    format="%b %Y",
    errors="coerce"
)

# Drop missing Month values explicitly
missing_months = df["Month_dt"].isna().sum()
if missing_months > 0:
    print(f"⚠️ Dropping {missing_months} records with missing Month")

df = df.dropna(subset=["Month_dt"]).copy()

# Filter from Jan 2025 onwards
df = df[df["Month_dt"] >= pd.Timestamp("2025-01-01")].copy()
print(f"Records after Month filter: {len(df)}")

# 2. FILTER: DUMPING & BURNING OFFENCES ONLY
dumping_burning_offences = [
    "Illegal dumping of Garbage on road sides/ vacant land",
    "Burning of garbage/plastic waste",
    "Burning of Biomass/garden waste"
]

df = df[df["Offences"].isin(dumping_burning_offences)].copy()
print(f"Records after offence filter: {len(df)}")

# 3. EXTRACT & CLEAN LATITUDE / LONGITUDE
df[["Latitude", "Longitude"]] = (
    df["Latitude & Longitude"]
    .astype(str)
    .str.replace(" ", "")
    .str.split(",", expand=True)
)

df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")

# Remove invalid coordinates
df["missing_latlon"] = df["Latitude"].isna() | df["Longitude"].isna()
df["zero_coordinates"] = (df["Latitude"] == 0) & (df["Longitude"] == 0)

df_valid = df[~df["missing_latlon"] & ~df["zero_coordinates"]].copy()
print(f"Records with valid coordinates: {len(df_valid)}")

# =====================================================
# 4. DBSCAN CLUSTERING (5 METERS)
# =====================================================
df_valid["lat_rad"] = np.radians(df_valid["Latitude"])
df_valid["lon_rad"] = np.radians(df_valid["Longitude"])

coords = df_valid[["lat_rad", "lon_rad"]].values

EARTH_RADIUS_M = 6_371_000
eps = 5 / EARTH_RADIUS_M  # 5 meters

db = DBSCAN(
    eps=eps,
    min_samples=1,
    metric="haversine"
)

df_valid["Cluster"] = db.fit_predict(coords)

# 5. HOTSPOT DIAGNOSTIC SUMMARY
cluster_counts = (
    df_valid.groupby("Cluster")["Compliant ID"]
    .nunique()
    .reset_index(name="Count")
)

single_complaints = cluster_counts.loc[
    cluster_counts["Count"] == 1, "Count"
].sum()

recurring_complaints = cluster_counts.loc[
    cluster_counts["Count"] >= 2, "Count"
].sum()

diagnostic_summary = pd.DataFrame({
    "Total dumping & burning complaints": [len(df_valid)],
    "Single-occurrence complaints": [single_complaints],
    "Recurring hotspot complaints (≥2)": [recurring_complaints],
    "Number of recurring hotspots": [
        (cluster_counts["Count"] >= 2).sum()
    ]
})

print("\n===== HOTSPOT DIAGNOSTIC =====")
print(diagnostic_summary)

# 6. KEEP ONLY RECURRING HOTSPOTS (≥ 2 complaints)
recurring_clusters = cluster_counts.loc[
    cluster_counts["Count"] >= 2, "Cluster"
]

df_recurring = df_valid[
    df_valid["Cluster"].isin(recurring_clusters)
].copy()

# 7. LOCATION-LEVEL SUMMARY
location_summary = (
    df_recurring.groupby("Cluster")
    .agg(
        Geo_Location=("Geo Location", "first"),
        Count=("Compliant ID", "nunique"),
        Latitude=("Latitude", "mean"),
        Longitude=("Longitude", "mean")
    )
    .reset_index()
)

# 8. OFFENCE TYPES WITH COUNTS
offence_counts = (
    df_recurring.groupby(["Cluster", "Offences"])["Compliant ID"]
    .nunique()
    .reset_index(name="cnt")
)

offence_text = (
    offence_counts
    .assign(
        offence_str=lambda x:
        x["Offences"] + " (" + x["cnt"].astype(str) + ")"
    )
    .groupby("Cluster")["offence_str"]
    .apply(", ".join)
    .reset_index(name="Offences")
)

# 9. FINAL HOTSPOT TABLE
final_table = (
    location_summary
    .merge(offence_text, on="Cluster", how="left")
    [["Geo_Location", "Count", "Offences", "Latitude", "Longitude"]]
)

final_table = (
    final_table
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)

final_table.insert(
    0,
    "Hotspot",
    [f"Hotspot {i+1}" for i in range(len(final_table))]
)


# 10. DISPLAY / EXPORT
pd.set_option("display.max_rows", None)
print("\n===== FIELD-READY HOTSPOTS =====")
print(final_table)

# Optional export
# download_path = os.path.join(
#     os.path.expanduser("~"),
#     "Downloads",
#     "msw_dumping_burning_hotspots_Jan2025_onwards.xlsx"
# )
# final_table.to_excel(download_path, index=False)
# print(f"\nExcel saved at: {download_path}")


***MSW Dumping-Burning Hotspots Jan2025_onwards with Ward***

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
import geopandas as gpd
from shapely.geometry import Point
import os

# 0. LOAD DATA
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path, encoding="latin1")

# 0a. CONVERT MONTH TO DATETIME
# Assuming the 'Month' column is in 'Jan 2025' format
df['Month_dt'] = pd.to_datetime(df['Month'], format='%b %Y', errors='coerce')

# Filter for Jan 2025 till now
start_date = pd.to_datetime('Jan 2025', format='%b %Y')
df = df[df['Month_dt'] >= start_date].copy()

print(f"Records after date filter: {len(df)}")

# 1. FILTER: DUMPING & BURNING OFFENCES
dumping_burning_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Burning of garbage/plastic waste',
    'Burning of Biomass/garden waste']

df = df[df['Offences'].isin(dumping_burning_offences)].copy()
print(f"Records after offence filter: {len(df)}")

# 2. EXTRACT & CLEAN LAT/LON
df[['Latitude', 'Longitude']] = df['Latitude & Longitude'].str.split(',', expand=True)
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# Flag missing or zero coordinates
df['missing_latlon'] = df['Latitude'].isna() | df['Longitude'].isna()
df['zero_coordinates'] = (df['Latitude'] == 0) & (df['Longitude'] == 0)

# Keep only valid coordinates
df_valid = df[~df['missing_latlon'] & ~df['zero_coordinates']].copy()
print(f"Records with valid coordinates: {len(df_valid)}")

# 3. DBSCAN CLUSTERING (5 METERS)
df_valid['lat_rad'] = np.radians(df_valid['Latitude'])
df_valid['lon_rad'] = np.radians(df_valid['Longitude'])
coords = df_valid[['lat_rad', 'lon_rad']].values

EARTH_RADIUS_M = 6_371_000
eps = 5 / EARTH_RADIUS_M  # 5 meters

db = DBSCAN(eps=eps, min_samples=1, metric='haversine').fit(coords)
df_valid['Cluster'] = db.labels_

# 4. HOTSPOT DIAGNOSTIC
cluster_counts = (
    df_valid.groupby('Cluster')['Compliant ID']
    .nunique()
    .reset_index(name='Count'))

single_complaints = cluster_counts.loc[cluster_counts['Count'] == 1, 'Count'].sum()
recurring_complaints = cluster_counts.loc[cluster_counts['Count'] >= 2, 'Count'].sum()
total_complaints = len(df_valid)

diagnostic_summary = pd.DataFrame({
    'Total dumping & burning complaints': [total_complaints],
    'Single-occurrence complaints': [single_complaints],
    'Recurring hotspot complaints (≥2)': [recurring_complaints],
    'Number of recurring hotspots': [len(cluster_counts[cluster_counts['Count'] >= 2])]})

print("\n===== HOTSPOT DIAGNOSTIC =====")
print(diagnostic_summary)

# 5. KEEP ONLY RECURRING HOTSPOTS (≥ 2 complaints)
recurring_clusters = cluster_counts.loc[cluster_counts['Count'] >= 2, 'Cluster']
df_recurring = df_valid[df_valid['Cluster'].isin(recurring_clusters)].copy()

# 6. LOCATION-LEVEL SUMMARY
location_summary = (
    df_recurring.groupby('Cluster')
    .agg(
        Geo_Location=('Geo Location', 'first'),
        Count=('Compliant ID', 'nunique'),
        Latitude=('Latitude', 'mean'),
        Longitude=('Longitude', 'mean')).reset_index())

# 7. OFFENCE TYPES WITH COUNTS
offence_counts = (
    df_recurring.groupby(['Cluster', 'Offences'])['Compliant ID']
    .nunique()
    .reset_index(name='cnt'))

offence_text = (
    offence_counts
    .assign(offence_str=lambda x: x['Offences'] + ' (' + x['cnt'].astype(str) + ')')
    .groupby('Cluster')['offence_str']
    .apply(', '.join)
    .reset_index(name='Offences'))

# 8. FINAL HOTSPOT TABLE
final_table = (
    location_summary.merge(offence_text, on='Cluster', how='left')
    [['Geo_Location', 'Count', 'Offences', 'Latitude', 'Longitude']])

# 8. SPATIAL JOIN: ADD WARD NAME

# Convert hotspots to GeoDataFrame
geometry = [Point(xy) for xy in zip(final_table['Longitude'], final_table['Latitude'])]

hotspot_gdf = gpd.GeoDataFrame(
    final_table,
    geometry=geometry,
    crs="EPSG:4326")

# Load Delhi Ward Shapefile
shp_path = r"C:\Users\Jeen priyee\Downloads\Delhi_Wards-SHP_Datameet\Delhi_Wards.shp"
delhi_gdf = gpd.read_file(shp_path)

# Ensure CRS match
delhi_gdf = delhi_gdf.to_crs("EPSG:4326")

# Spatial join
hotspot_with_ward = gpd.sjoin(
    hotspot_gdf,
    delhi_gdf,
    how="left",
    predicate="within")

# 🔴 CHANGE THIS COLUMN NAME IF NEEDED
WARD_COL = "WARD_NAME"   # <-- inspect shapefile if this errors

hotspot_with_ward = hotspot_with_ward.rename(
    columns={WARD_COL: "Ward Name"})

# 9. FINAL CLEAN TABLE
final_output = (
    hotspot_with_ward
    .drop(columns=['geometry', 'index_right'])
    .sort_values(by='Count', ascending=False)
    .reset_index(drop=True))

final_output.insert(
    0,
    'Hotspot',
    ['Hotspot ' + str(i + 1) for i in range(len(final_output))])


# 10. EXPORT
#download_path = os.path.join(os.path.expanduser("~"), "Downloads", "msw_dumping_burning_hotspots_Jan2025_onwards_with_wards.xlsx")
#final_output.to_excel(download_path, index=False)

pd.set_option("display.max_rows", None)
print("\n===== FIELD-READY HOTSPOTS =====")
print(final_output)
#print("\nExcel saved at:", download_path)




***MSW_dumping_hotspots_Jan2025_onwards***

import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
import os

# 0. LOAD DATA
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path, encoding="latin1")

# 0a. CONVERT MONTH TO DATETIME
# Assuming the 'Month' column is in 'Jan 2025' format
df['Month_dt'] = pd.to_datetime(df['Month'], format='%b %Y', errors='coerce')

# Filter for Jan 2025 till now
start_date = pd.to_datetime('Jan 2025', format='%b %Y')
df = df[df['Month_dt'] >= start_date].copy()

print(f"Records after date filter: {len(df)}")

# 1. FILTER: DUMPING & BURNING OFFENCES
dumping_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land']

df = df[df['Offences'].isin(dumping_offences)].copy()
print(f"Records after offence filter: {len(df)}")

# 2. EXTRACT & CLEAN LAT/LON
df[['Latitude', 'Longitude']] = df['Latitude & Longitude'].str.split(',', expand=True)
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# Flag missing or zero coordinates
df['missing_latlon'] = df['Latitude'].isna() | df['Longitude'].isna()
df['zero_coordinates'] = (df['Latitude'] == 0) & (df['Longitude'] == 0)

# Keep only valid coordinates
df_valid = df[~df['missing_latlon'] & ~df['zero_coordinates']].copy()
print(f"Records with valid coordinates: {len(df_valid)}")

# 3. DBSCAN CLUSTERING (5 METERS)
df_valid['lat_rad'] = np.radians(df_valid['Latitude'])
df_valid['lon_rad'] = np.radians(df_valid['Longitude'])
coords = df_valid[['lat_rad', 'lon_rad']].values

EARTH_RADIUS_M = 6_371_000
eps = 5 / EARTH_RADIUS_M  # 5 meters

db = DBSCAN(eps=eps, min_samples=1, metric='haversine').fit(coords)
df_valid['Cluster'] = db.labels_

# 4. HOTSPOT DIAGNOSTIC
cluster_counts = (
    df_valid.groupby('Cluster')['Compliant ID']
    .nunique()
    .reset_index(name='Count'))

single_complaints = cluster_counts.loc[cluster_counts['Count'] == 1, 'Count'].sum()
recurring_complaints = cluster_counts.loc[cluster_counts['Count'] >= 2, 'Count'].sum()
total_complaints = len(df_valid)

diagnostic_summary = pd.DataFrame({
    'Total dumping & burning complaints': [total_complaints],
    'Single-occurrence complaints': [single_complaints],
    'Recurring hotspot complaints (≥2)': [recurring_complaints],
    'Number of recurring hotspots': [len(cluster_counts[cluster_counts['Count'] >= 2])]})

print("\n===== HOTSPOT DIAGNOSTIC =====")
print(diagnostic_summary)

# 5. KEEP ONLY RECURRING HOTSPOTS (≥ 2 complaints)
recurring_clusters = cluster_counts.loc[cluster_counts['Count'] >= 2, 'Cluster']
df_recurring = df_valid[df_valid['Cluster'].isin(recurring_clusters)].copy()

# 6. LOCATION-LEVEL SUMMARY
location_summary = (
    df_recurring.groupby('Cluster')
    .agg(
        Geo_Location=('Geo Location', 'first'),
        Count=('Compliant ID', 'nunique'),
        Latitude=('Latitude', 'mean'),
        Longitude=('Longitude', 'mean')).reset_index())

# 7. OFFENCE TYPES WITH COUNTS
offence_counts = (
    df_recurring.groupby(['Cluster', 'Offences'])['Compliant ID']
    .nunique()
    .reset_index(name='cnt'))

offence_text = (
    offence_counts
    .assign(offence_str=lambda x: x['Offences'] + ' (' + x['cnt'].astype(str) + ')')
    .groupby('Cluster')['offence_str']
    .apply(', '.join)
    .reset_index(name='Offences'))

# 8. FINAL HOTSPOT TABLE
final_table = (
    location_summary.merge(offence_text, on='Cluster', how='left')
    [['Geo_Location', 'Count', 'Offences', 'Latitude', 'Longitude']])

# Sort and label hotspots
final_table_sorted = final_table.sort_values(by='Count', ascending=False).reset_index(drop=True)
final_table_sorted.insert(0, 'Hotspot', ['Hotspot ' + str(i + 1) for i in range(len(final_table_sorted))])

# 9. EXPORT
#download_path = os.path.join(os.path.expanduser("~"), "Downloads", "msw_dumping_hotspots_Jan2025_onwards.xlsx")
#final_table_sorted.to_excel(download_path, index=False)

pd.set_option("display.max_rows", None)
print("\n===== FIELD-READY HOTSPOTS =====")
print(final_table_sorted)
#print("\nExcel saved at:", download_path)


***MSW_dumping_hotspots_Jan2025_onwards_with_Ward***

import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
import geopandas as gpd
from shapely.geometry import Point
import os

# 0. LOAD DATA
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
df = pd.read_csv(csv_path, encoding="latin1")

# 0a. CONVERT MONTH TO DATETIME
# Assuming the 'Month' column is in 'Jan 2025' format
df['Month_dt'] = pd.to_datetime(df['Month'], format='%b %Y', errors='coerce')

# Filter for Jan 2025 till now
start_date = pd.to_datetime('Jan 2025', format='%b %Y')
df = df[df['Month_dt'] >= start_date].copy()

print(f"Records after date filter: {len(df)}")

# 1. FILTER: DUMPING & BURNING OFFENCES
dumping_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land']

df = df[df['Offences'].isin(dumping_offences)].copy()
print(f"Records after offence filter: {len(df)}")

# 2. EXTRACT & CLEAN LAT/LON
df[['Latitude', 'Longitude']] = df['Latitude & Longitude'].str.split(',', expand=True)
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# Flag missing or zero coordinates
df['missing_latlon'] = df['Latitude'].isna() | df['Longitude'].isna()
df['zero_coordinates'] = (df['Latitude'] == 0) & (df['Longitude'] == 0)

# Keep only valid coordinates
df_valid = df[~df['missing_latlon'] & ~df['zero_coordinates']].copy()
print(f"Records with valid coordinates: {len(df_valid)}")

# 3. DBSCAN CLUSTERING (5 METERS)
df_valid['lat_rad'] = np.radians(df_valid['Latitude'])
df_valid['lon_rad'] = np.radians(df_valid['Longitude'])
coords = df_valid[['lat_rad', 'lon_rad']].values

EARTH_RADIUS_M = 6_371_000
eps = 5 / EARTH_RADIUS_M  # 5 meters

db = DBSCAN(eps=eps, min_samples=1, metric='haversine').fit(coords)
df_valid['Cluster'] = db.labels_

# 4. HOTSPOT DIAGNOSTIC
cluster_counts = (
    df_valid.groupby('Cluster')['Compliant ID']
    .nunique()
    .reset_index(name='Count'))

single_complaints = cluster_counts.loc[cluster_counts['Count'] == 1, 'Count'].sum()
recurring_complaints = cluster_counts.loc[cluster_counts['Count'] >= 2, 'Count'].sum()
total_complaints = len(df_valid)

diagnostic_summary = pd.DataFrame({
    'Total dumping & burning complaints': [total_complaints],
    'Single-occurrence complaints': [single_complaints],
    'Recurring hotspot complaints (≥2)': [recurring_complaints],
    'Number of recurring hotspots': [len(cluster_counts[cluster_counts['Count'] >= 2])]})

print("\n===== HOTSPOT DIAGNOSTIC =====")
print(diagnostic_summary)

# 5. KEEP ONLY RECURRING HOTSPOTS (≥ 2 complaints)
recurring_clusters = cluster_counts.loc[cluster_counts['Count'] >= 2, 'Cluster']
df_recurring = df_valid[df_valid['Cluster'].isin(recurring_clusters)].copy()

# 6. LOCATION-LEVEL SUMMARY
location_summary = (
    df_recurring.groupby('Cluster')
    .agg(
        Geo_Location=('Geo Location', 'first'),
        Count=('Compliant ID', 'nunique'),
        Latitude=('Latitude', 'mean'),
        Longitude=('Longitude', 'mean')).reset_index())

# 7. OFFENCE TYPES WITH COUNTS
offence_counts = (
    df_recurring.groupby(['Cluster', 'Offences'])['Compliant ID']
    .nunique()
    .reset_index(name='cnt'))

offence_text = (
    offence_counts
    .assign(offence_str=lambda x: x['Offences'] + ' (' + x['cnt'].astype(str) + ')')
    .groupby('Cluster')['offence_str']
    .apply(', '.join)
    .reset_index(name='Offences'))

# 8. FINAL HOTSPOT TABLE
final_table = (
    location_summary.merge(offence_text, on='Cluster', how='left')
    [['Geo_Location', 'Count', 'Offences', 'Latitude', 'Longitude']])

# 8. SPATIAL JOIN: ADD WARD NAME

# Convert hotspots to GeoDataFrame
geometry = [Point(xy) for xy in zip(final_table['Longitude'], final_table['Latitude'])]

hotspot_gdf = gpd.GeoDataFrame(
    final_table,
    geometry=geometry,
    crs="EPSG:4326")

# Load Delhi Ward Shapefile
shp_path = r"C:\Users\Jeen priyee\Downloads\Delhi_Wards-SHP_Datameet\Delhi_Wards.shp"
delhi_gdf = gpd.read_file(shp_path)

# Ensure CRS match
delhi_gdf = delhi_gdf.to_crs("EPSG:4326")

# Spatial join
hotspot_with_ward = gpd.sjoin(
    hotspot_gdf,
    delhi_gdf,
    how="left",
    predicate="within")

# 🔴 CHANGE THIS COLUMN NAME IF NEEDED
WARD_COL = "WARD_NAME"   # <-- inspect shapefile if this errors

hotspot_with_ward = hotspot_with_ward.rename(
    columns={WARD_COL: "Ward Name"})

# 9. FINAL CLEAN TABLE
final_output = (
    hotspot_with_ward
    .drop(columns=['geometry', 'index_right'])
    .sort_values(by='Count', ascending=False)
    .reset_index(drop=True))

final_output.insert(
    0,
    'Hotspot',
    ['Hotspot ' + str(i + 1) for i in range(len(final_output))])


# 10. EXPORT
#download_path = os.path.join(os.path.expanduser("~"), "Downloads", "msw_dumping_hotspots_Jan2025_onwards_with_wards.xlsx")
#final_output.to_excel(download_path, index=False)

pd.set_option("display.max_rows", None)
print("\n===== FIELD-READY HOTSPOTS =====")
print(final_output)
#print("\nExcel saved at:", download_path)




***Common GVP's From the entire and recent***

In [ ]:
import pandas as pd
import os

# 1. LOAD BOTH HOTSPOT FILES

citywide_path = r"C:\Users\Jeen priyee\Downloads\msw_dumping_burning\msw_dumping_burning_citywide_hotspots_with_ward.xlsx"
recent_path = r"C:\Users\Jeen priyee\Downloads\msw_dumping_burning\msw_dumping_burning_hotspots_Jan2025_onwards_with_wards.xlsx"

citywide = pd.read_excel(citywide_path)
recent = pd.read_excel(recent_path)

print("Citywide hotspots:", len(citywide))
print("Recent hotspots (Jan 2025+):", len(recent))

# 2. STANDARDISE GEO LOCATION (CRITICAL)

citywide["Geo_Location_clean"] = (
    citywide["Geo_Location"].astype(str).str.strip().str.lower()
)

recent["Geo_Location_clean"] = (
    recent["Geo_Location"].astype(str).str.strip().str.lower()
)

# 3. FIND COMMON (PERSISTENT) HOTSPOTS

common_hotspots = citywide.merge(
    recent,
    on="Geo_Location_clean",
    how="inner",
    suffixes=("_citywide", "_recent"))

print("Persistent hotspots found:", len(common_hotspots))

# 4. SAFELY HANDLE WARD NAME

if "Ward Name_recent" in common_hotspots.columns:
    common_hotspots["Ward Name"] = common_hotspots["Ward Name_recent"]
elif "Ward Name_citywide" in common_hotspots.columns:
    common_hotspots["Ward Name"] = common_hotspots["Ward Name_citywide"]
else:
    common_hotspots["Ward Name"] = None

# 5. FINAL FIELD-READY TABLE

final_common_hotspots = (
    common_hotspots[
        [
            "Geo_Location_citywide",
            "Ward Name",
            "Count_citywide",
            "Offences_citywide",
            "Count_recent",
            "Offences_recent",
            "Latitude_recent",
            "Longitude_recent",]].rename(
        columns={
            "Geo_Location_citywide": "Geo Location",
            "Count_citywide": "Total Complaints (All Time)",
            "Offences_citywide": "Offences (All Time)",
            "Count_recent": "Complaints Since Jan 2025",
            "Offences_recent": "Recent Offences",
            "Latitude_recent": "Latitude",
            "Longitude_recent": "Longitude",})
    .sort_values(by="Complaints Since Jan 2025", ascending=False)
    .reset_index(drop=True))

# 6. ADD HOTSPOT LABEL

final_common_hotspots.insert(0, "Hotspot", ["Hotspot " + str(i + 1) for i in range(len(final_common_hotspots))])

# 7. EXPORT TO EXCEL

# output_path = os.path.join(os.path.expanduser("~"), "Downloads","persistent_dumping_burning_hotspots_citywide_and_recent.xlsx",)

#final_common_hotspots.to_excel(output_path, index=False)

# 8. DISPLAY OUTPUT

pd.set_option("display.max_rows", None)

print("\n===== PERSISTENT HOTSPOTS (CITYWIDE ∩ JAN 2025+) =====")
print(final_common_hotspots)

#print("\nExcel saved at:", output_path))

In [ ]:
import pandas as pd
import os
import re

# 1. FILE PATHS and LOAD FILES

citywide_path = r"C:\Users\Jeen priyee\Downloads\msw_dumping_burning\msw_dumping_burning_citywide_hotspots_with_ward.xlsx"
recent_path = r"C:\Users\Jeen priyee\Downloads\msw_dumping_burning\msw_dumping_burning_hotspots_Jan2025_onwards_with_wards.xlsx"

citywide = pd.read_excel(citywide_path)
recent = pd.read_excel(recent_path)

# 2. REQUIRED COLUMN CHECK (MINIMAL & SAFE)

required_cols = {
    "Geo_Location", "Latitude", "Longitude", "Count", "Offences", "Ward_Name"}

for name, df in {
    "Citywide": citywide,
    "Recent": recent
}.items():
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"{name} file missing columns: {missing}")

# 3. ROBUST GEO LOCATION CLEANING

def clean_geo(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

citywide["Geo_clean"] = citywide["Geo_Location"].apply(clean_geo)
recent["Geo_clean"] = recent["Geo_Location"].apply(clean_geo)

# 4. BASIC COUNTS

citywide_count = citywide["Geo_clean"].nunique()
recent_count = recent["Geo_clean"].nunique()

# 5. FIND PERSISTENT GVPs

persistent_geo = set(citywide["Geo_clean"]).intersection(
    set(recent["Geo_clean"]))

persistent_count = len(persistent_geo)

citywide_common = citywide[citywide["Geo_clean"].isin(persistent_geo)]
recent_common = recent[recent["Geo_clean"].isin(persistent_geo)]

# 6. COMBINE & AGGREGATE (ONE ROW PER GVP)

combined = pd.concat(
    [citywide_common, recent_common],
    ignore_index=True)

final = (
    combined
    .groupby("Geo_clean")
    .agg(
        GVP_Location=("Geo_Location", "first"),

        Offences=(
            "Offences",
            lambda x: ", ".join(
                sorted(set(
                    ", ".join(x.dropna()).split(", ")
                )))),

        Ward_Name=(
            "Ward_Name",
            lambda x: ", ".join(sorted(set(x.dropna())))),

        Latitude=("Latitude", "median"),
        Longitude=("Longitude", "median"),
        Total_Complaints=("Count", "sum")).reset_index(drop=True))

# 7. SORT & LABEL GVPs

final_sorted = (
    final
    .sort_values(by="Total_Complaints", ascending=False)
    .reset_index(drop=True))

final_sorted.insert(
    0,
    "GVP_ID",
    ["GVP " + str(i + 1) for i in range(len(final_sorted))])

final_output = final_sorted[
    [
        "GVP_ID",
        "GVP_Location",
        "Ward_Name",
        "Offences",
        "Latitude",
        "Longitude",
        "Total_Complaints"]]

# 8. SUMMARY TABLE

summary_df = pd.DataFrame({
    "Metric": [
        "Citywide GVPs",
        "Recent GVPs (Jan 2025+)",
        "Persistent GVPs"],
    "Count": [
        citywide_count,
        recent_count,
        persistent_count]})

# 9. EXPORT (MULTI-SHEET EXCEL)

output_path = os.path.join(
    os.path.expanduser("~"),
    "Downloads",
    "recurring_GVPs_dumping_burning_Jan2025_onwards.xlsx")

with pd.ExcelWriter(output_path) as writer:
    final_output.to_excel(
        writer,
        sheet_name="recurring_GVPs",
        index=False)
    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False)

# 10. DISPLAY SUMMARY

print("\n===== GVP SUMMARY =====")
print(f"Citywide GVPs: {citywide_count}")
print(f"Recent GVPs (Jan 2025+): {recent_count}")
print(f"Persistent GVPs found: {persistent_count}")

print("\nExcel saved at:")
print(output_path)


In [ ]:
#Chloropleth map that shows pending complaint count of all the wards highlighting top 5 wards
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import Point, box
from shapely.geometry.polygon import Polygon
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.patches as mpatches

# Load the CSV file with a specified encoding
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"

complaints_df = pd.read_csv(csv_path, encoding='ISO-8859-1')

# Split latitude and longitude into separate columns
complaints_df[['Latitude', 'Longitude']] = complaints_df['Latitude & Longitude'].str.split(',', expand=True)
complaints_df['Latitude'] = complaints_df['Latitude'].astype(float)
complaints_df['Longitude'] = complaints_df['Longitude'].astype(float)

# Filter for pending complaints
pending_complaints_df = complaints_df[complaints_df['Status'] == 'Pending']

# Load the shapefile
shp_path = r'C:\Users\Jeen priyee\Downloads\Delhi_Wards-SHP_Datameet\Delhi_Wards.shp'

delhi_gdf = gpd.read_file(shp_path)

# Create GeoDataFrame for pending complaints
geometry = [Point(xy) for xy in zip(pending_complaints_df['Longitude'], pending_complaints_df['Latitude'])]
pending_complaints_gdf = gpd.GeoDataFrame(pending_complaints_df, geometry=geometry)

# Set CRS for both GeoDataFrames if needed
if delhi_gdf.crs is None:
    delhi_gdf.set_crs(epsg=4326, inplace=True)
    
pending_complaints_gdf.crs = delhi_gdf.crs

# Perform a spatial join to aggregate occurrences by region
joined_gdf = gpd.sjoin(pending_complaints_gdf, delhi_gdf, how='left', predicate='within')

# Count occurrences in each region
region_counts = joined_gdf.groupby('index_right').size()

# Get ward name information if available in the shapefile
if 'name' in delhi_gdf.columns:
    ward_name_col = 'name'
elif 'NAME' in delhi_gdf.columns:
    ward_name_col = 'NAME'
elif 'Ward_Name' in delhi_gdf.columns:
    ward_name_col = 'Ward_Name'
elif 'WARD_NAME' in delhi_gdf.columns:
    ward_name_col = 'WARD_NAME'
else:
    ward_name_col = None
    print("No ward name column found, using index as identifier")

# Create a DataFrame with ward information and counts
if ward_name_col:
    ward_info = delhi_gdf[[ward_name_col]].copy()
    ward_info['occurrences'] = region_counts.reindex(delhi_gdf.index, fill_value=0)
    
    # Sort by occurrences to find top 5
    top_5_wards = ward_info.sort_values('occurrences', ascending=False).head(5)
    print("\nTop 5 Wards by Number of Pending Complaints:")
    print(top_5_wards)
else:
    # Use index if no name column is available
    ward_info = pd.DataFrame(index=delhi_gdf.index)
    ward_info['occurrences'] = region_counts.reindex(delhi_gdf.index, fill_value=0)
    
    # Sort by occurrences to find top 5
    top_5_wards = ward_info.sort_values('occurrences', ascending=False).head(5)
    print("\nTop 5 Wards by Number of Pending Complaints (using index):")
    print(top_5_wards)

# Merge the counts back to the GeoDataFrame
delhi_gdf['occurrences'] = region_counts.reindex(delhi_gdf.index, fill_value=0)

# Reset the index to ensure 'index' is available as a column
delhi_gdf = delhi_gdf.reset_index()

# Print occurrences for validation
print("\nOccurrences in each zone (Pending Complaints):")
print(delhi_gdf[['index', 'occurrences']])

# Create Folium map
m = folium.Map(location=[28.7041, 77.1025], zoom_start=11)

# Create a larger background rectangle (much bigger than Delhi's bounds)
min_x, min_y, max_x, max_y = delhi_gdf.total_bounds
padding = 1
background_box = box(
    min_x - padding,
    min_y - padding,
    max_x + padding,
    max_y + padding
)

# Create the mask by subtracting Delhi's shape from the background
delhi_union = delhi_gdf.geometry.unary_union
mask = background_box.difference(delhi_union)

# Add the white background mask to the map
folium.GeoJson(
    mask.__geo_interface__,
    style_function=lambda x: {
        'fillColor': 'white',
        'color': 'white',
        'fillOpacity': 1,
        'weight': 0
    }
).add_to(m)

# Add the choropleth layer
folium.Choropleth(
    geo_data=delhi_gdf.__geo_interface__,
    data=delhi_gdf,
    columns=['index', 'occurrences'],
    key_on='feature.properties.index',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Number of Pending Complaints'
).add_to(m)

# Save the map to an HTML file
m.save('delhi_pending_complaints_map.html')

# Now create a static map with matplotlib for JPEG export
# Create a figure with white background
fig, ax = plt.subplots(figsize=(20, 5), facecolor='white')
ax.set_facecolor('white')

# Create a colormap similar to YlOrRd in Folium
colors_list = ['#ffffcc', '#ffeda0', '#fed976', '#feb24c', '#fd8d3c', '#fc4e2a', '#e31a1c', '#bd0026', '#800026']
cmap = colors.LinearSegmentedColormap.from_list('YlOrRd', colors_list)

# Plot the choropleth
delhi_gdf.plot(
    column='occurrences',
    cmap=cmap,
    linewidth=0.8,
    edgecolor='black',
    alpha=0.7,
    ax=ax
)

# Remove axis
ax.set_axis_off()

# Get the current axis limits
x_min, x_max = ax.get_xlim()
y_min, y_max = ax.get_ylim()

# Add padding to ensure the white background covers everything
padding_factor = 0.1
width = x_max - x_min
height = y_max - y_min
x_min -= width * padding_factor
x_max += width * padding_factor
y_min -= height * padding_factor
y_max += height * padding_factor

# Reset the axis limits with padding
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

# Add title
plt.title('Delhi Pending Complaints Map', fontsize=20)

# Create a custom legend
# Get the min and max values for the colorbar
vmin = delhi_gdf['occurrences'].min()
vmax = delhi_gdf['occurrences'].max()

# Create a colorbar separately
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])  # You need to set an array for the ScalarMappable
cbar = plt.colorbar(sm, ax=ax, orientation='horizontal', fraction=0.046, pad=0.04)
cbar.set_label('Number of Pending Complaints', fontsize=14)

# Optional: Highlight the top 5 wards on the map
if len(top_5_wards) > 0:
    # Get original indices of top 5 wards
    top_indices = top_5_wards.index.tolist()
    
    # Find these in the reset index dataframe
    for idx in top_indices:
        # Find the matching row in delhi_gdf
        ward_row = delhi_gdf[delhi_gdf['index'] == idx]
        if not ward_row.empty:
            # Add a thicker outline to highlight this ward
            ward_row.boundary.plot(ax=ax, color='black', linewidth=2.5)

# Save as JPEG with tight layout and white background
plt.savefig('delhi_pending_complaints_map.jpg', 
            dpi=300, 
            bbox_inches='tight', 
            facecolor='white', 
            edgecolor='none')

print("\nMap saved as delhi_pending_complaints_map.jpg")

# Show the map
plt.show()

In [ ]:
#Code to get chloropleth of pending cases for diffrent offences  
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import Point, Polygon
import os
import matplotlib.pyplot as plt
from folium.plugins import MarkerCluster
from branca.element import Template, MacroElement
from matplotlib.colors import LinearSegmentedColormap
import contextily as ctx
import numpy as np

# Load the CSV file with a specified encodin

# Create output directory if it doesn't exist
output_dir = r'C:\Users\Jeen priyee\Downloads\gda_new_2026\pending'
os.makedirs(output_dir, exist_ok=True)

# Get list of all unique offenses
offenses = complaints_df['Offences'].unique()

# Create a dictionary to store all ward counts for the master CSV
master_ward_counts = {}

# Process each offense
for offense in offenses:
    # Filter complaints for the specific offense and status
    status_filter = "Pending"
    filtered_complaints_df = complaints_df[(complaints_df['Offences'] == offense) & 
                                          (complaints_df['Status'] == status_filter)]
    
    # Skip if no records found for this offense
    if filtered_complaints_df.empty:
        print(f"No pending complaints found for offense: {offense}")
        continue
    
    # Create GeoDataFrame for filtered complaints
    geometry = [Point(xy) for xy in zip(filtered_complaints_df['Longitude'], filtered_complaints_df['Latitude'])]
    filtered_complaints_gdf = gpd.GeoDataFrame(filtered_complaints_df, geometry=geometry)
    
    # Ensure both GeoDataFrames use the same coordinate reference system (CRS)
    filtered_complaints_gdf.crs = delhi_gdf.crs
    
    # Perform a spatial join to aggregate occurrences by region
    joined_gdf = gpd.sjoin(filtered_complaints_gdf, delhi_gdf, how='left', predicate='within')
    
    # Count occurrences in each region
    region_counts = joined_gdf.groupby('index_right').size()
    
    # Create a DataFrame for ward-level counts for this offense
    ward_counts_df = pd.DataFrame({
        'Ward_ID': region_counts.index,
        'Ward_Name': [delhi_gdf.loc[idx, 'Ward_Name'] if idx in delhi_gdf.index else 'Unknown' for idx in region_counts.index],
        'Count': region_counts.values,
        'Offense': offense
    })
    
    # Create a safe filename from the offense
    safe_filename = "".join([c if c.isalnum() else "_" for c in offense])
    
    # Save individual CSV for this offense with counts for all wards
    ward_counts_file = os.path.join(output_dir, f'ward_counts_{safe_filename}.csv')
    ward_counts_df.to_csv(ward_counts_file, index=False)
    print(f"Saved ward counts for {offense} to {ward_counts_file}")
    
    # Update the master counts dictionary
    for idx, row in ward_counts_df.iterrows():
        ward_id = row['Ward_ID']
        ward_name = row['Ward_Name']
        
        if ward_id not in master_ward_counts:
            master_ward_counts[ward_id] = {'Ward_ID': ward_id, 'Ward_Name': ward_name}
        
        # Add this offense's count to the master dictionary
        master_ward_counts[ward_id][offense] = row['Count']
    
    # Merge the counts back to the GeoDataFrame
    delhi_gdf_with_counts = delhi_gdf.copy()
    delhi_gdf_with_counts['occurrences'] = region_counts.reindex(delhi_gdf.index, fill_value=0)
    
    # Reset the index to ensure 'index' is available as a column
    delhi_gdf_with_counts = delhi_gdf_with_counts.reset_index()
    
    # Prepare the GeoJSON data with updated properties
    geojson_data = delhi_gdf_with_counts.__geo_interface__
    
    # Update occurrences in each zone within the GeoJSON data
    for feature in geojson_data['features']:
        zone_id = feature['id']
        if zone_id in region_counts.index:
            feature['properties']['occurrences'] = int(region_counts.loc[zone_id])
        else:
            feature['properties']['occurrences'] = 0
    
    # Extract and save the top 5 wards with the highest occurrences
    top_5_wards = delhi_gdf_with_counts.nlargest(5, 'occurrences')[['Ward_Name', 'occurrences']]
    
    # Save the top 5 table as a CSV
    top_5_wards.to_csv(os.path.join(output_dir, f'top5_wards_{safe_filename}.csv'), index=False)
    
    # Also save as an HTML table for better visualization
    html_table = top_5_wards.to_html(index=False)
    with open(os.path.join(output_dir, f'top5_wards_{safe_filename}.html'), 'w') as f:
        f.write(f"<h2>Top 5 Wards for {offense} (Pending)</h2>\n")
        f.write(html_table)
    
    # Create a folium map centered around Delhi
    m = folium.Map(location=[28.7041, 77.1025], zoom_start=11)
    
    # Add the choropleth layer
    choropleth = folium.Choropleth(
        geo_data=geojson_data,
        data=delhi_gdf_with_counts,
        columns=['index', 'occurrences'],
        key_on='feature.properties.index',
        fill_color='YlOrRd',
        fill_opacity=0.7,
        line_opacity=0.2,
        legend_name=f'Number of Pending Occurrences: {offense}'
    ).add_to(m)
    
    # Define the bounding box for the shapefile area
    bounds = delhi_gdf.total_bounds  # [minx, miny, maxx, maxy]
    
    # Create a large white polygon covering the entire map area
    map_bounds = Polygon([
        [bounds[0] - 1, bounds[1] - 1],
        [bounds[0] - 1, bounds[3] + 1],
        [bounds[2] + 1, bounds[3] + 1],
        [bounds[2] + 1, bounds[1] - 1]
    ])
    
    # Subtract the shapefile area to create the mask
    delhi_union = delhi_gdf.geometry.union_all()
    mask = gpd.GeoSeries(map_bounds).difference(delhi_union)

    
    # Add the mask to the map as a white overlay
    for geom in mask:
        folium.GeoJson(
            geom.__geo_interface__,
            style_function=lambda x: {
                'fillColor': 'white',
                'color': 'white',
                'fillOpacity': 1,
                'weight': 0
            }
        ).add_to(m)
    
    # Add tooltip to show ward name and count when hovering
    folium.GeoJson(
        geojson_data,
        style_function=lambda x: {'fillOpacity': 0, 'color': 'transparent'},
        tooltip=folium.GeoJsonTooltip(
            fields=['Ward_Name', 'occurrences'],
            aliases=['Ward Name:', 'Pending Cases:'],
            style=("background-color: white; color: #333; font-family: arial; font-size: 12px; padding: 10px;")
        )
    ).add_to(m)
    
    # Create a feature group for the markers
    marker_group = folium.FeatureGroup(name="Pending Cases")
    
    # Add markers for each complaint location
    for idx, row in filtered_complaints_df.iterrows():
        # Create a pop-up with relevant information
        popup_text = f"""
        <strong>Offense:</strong> {row['Offences']}<br>
        <strong>Zone:</strong> {row.get('Zone', 'Unknown')}<br>
        <strong>Status:</strong> {row['Status']}<br>
        """
        
        # Add a small circle marker instead of a standard marker for better performance
        folium.CircleMarker(
            location=[row['Latitude'], row['Longitude']],
            radius=0.52,  # Very small radius
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.7,
            popup=folium.Popup(popup_text, max_width=300),
            tooltip=f"Case ID: {row.get('Complaint_ID', idx)}"
        ).add_to(marker_group)
    
    # Add the marker group to the map
    marker_group.add_to(m)
    
    # Add layer control to toggle between markers and choropleth
    folium.LayerControl().add_to(m)
    
    # Save the map to an HTML file
    map_file = os.path.join(output_dir, f'delhi_Pending_{safe_filename}.html')
    m.save(map_file)
    
    # Now modify the HTML file to center the legend
    with open(map_file, 'r', encoding='utf-8') as file:
        html_content = file.read()
    
    # Modify the legend CSS to position it in the center
    # Find the .legend CSS block and modify it
    legend_css_original = """
            .legend {
                line-height: 18px;
                color: #555;
            }
            .legend i {
                width: 18px;
                height: 18px;
                float: left;
                margin-right: 8px;
                opacity: 0.7;
            }
    """
    
    legend_css_centered = """
            .legend {
                line-height: 18px;
                color: #555;
                position: absolute;
                z-index: 1000;
                left: 50%;
                transform: translateX(-50%);
                bottom: 20px;
                background-color: white;
                padding: 10px;
                border-radius: 5px;
                box-shadow: 0 0 15px rgba(0,0,0,0.2);
            }
            .legend i {
                width: 18px;
                height: 18px;
                float: left;
                margin-right: 8px;
                opacity: 0.7;
            }
    """
    
    # Replace the original legend CSS with our centered version
    html_content = html_content.replace(legend_css_original, legend_css_centered)
    
    # Write the modified HTML back to the file
    with open(map_file, 'w', encoding='utf-8') as file:
        file.write(html_content)
    
    print(f"Processed: {offense} - Map with markers and centered legend saved")
    # Create a matplotlib figure for static map export
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    
    # Create a custom colormap similar to 'YlOrRd'
    colors = ['#ffffb2', '#fed976', '#feb24c', '#fd8d3c', '#f03b20', '#bd0026']
    cmap = LinearSegmentedColormap.from_list('custom_YlOrRd', colors)
    
    # Plot the choropleth map
    delhi_gdf_with_counts.plot(
        column='occurrences',
        ax=ax,
        cmap=cmap,
        edgecolor='#666666',
        linewidth=0.5,
        legend=True,
        legend_kwds={
            'label': f'Number of Pending Occurrences: {offense}',
            'orientation': 'horizontal',
            'shrink': 0.8,
            'pad': 0.01,
            'aspect': 25
        }
    )
    
    # Add basemap for context (optional)
    try:
        ctx.add_basemap(ax, crs=delhi_gdf.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik)
    except Exception as e:
        print(f"Basemap couldn't be added - continuing without it: {e}")
    
    # Filter points to only include those within Delhi boundaries
    delhi_boundary = delhi_gdf.geometry.union_all()

    points_within_delhi = filtered_complaints_gdf[filtered_complaints_gdf.intersects(delhi_boundary)]
    print(f"Filtered out {len(filtered_complaints_gdf) - len(points_within_delhi)} points outside Delhi boundary")
    
    # Plot markers for each complaint location that is within Delhi boundaries
    if not points_within_delhi.empty:
        points_within_delhi.plot(
            ax=ax,
            color='blue',
            markersize=5,
            alpha=0.7,
            zorder=5  # Make sure points appear on top
        )
        
        # Add a legend for the markers
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', 
                   markersize=8, label='Pending Complaints', alpha=0.7)
        ]
        ax.legend(handles=legend_elements, loc='upper right', frameon=True, 
                  facecolor='white', edgecolor='gray')
    else:
        print(f"No points within Delhi boundary for {offense}")
    
    # Set the map extent to exactly match the Delhi boundaries
    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    
    # Improve the appearance
    ax.set_title(f'Delhi Wards - Pending {offense} Cases', fontsize=16)
    ax.set_axis_off()  # Remove axis
    plt.tight_layout()
    
    # Save as JPEG
    jpeg_file = os.path.join(output_dir, f'delhi_Pending_{safe_filename}.jpg')
    plt.savefig(jpeg_file, format='jpeg', dpi=300, bbox_inches='tight')
    plt.close()  # Close the figure to free memory
    
    print(f"Static JPEG map with filtered markers saved for {offense}")

# Create and save the master CSV with all offenses for all wards
# First, ensure all wards from the shapefile are included in the master counts
for idx, row in delhi_gdf.iterrows():
    ward_id = idx
    ward_name = row['Ward_Name']
    
    if ward_id not in master_ward_counts:
        master_ward_counts[ward_id] = {'Ward_ID': ward_id, 'Ward_Name': ward_name}

# Convert master dictionary to DataFrame
master_df = pd.DataFrame.from_dict(master_ward_counts.values())

# Fill NaN values with zeros (wards with no complaints for a particular offense)
master_df = master_df.fillna(0)

# Calculate total pending complaints across all offense types
master_df['Total_Pending'] = master_df.drop(['Ward_ID', 'Ward_Name'], axis=1).sum(axis=1)

# Sort by total pending complaints
master_df = master_df.sort_values('Total_Pending', ascending=False)

# Save the master CSV
master_csv_path = os.path.join(output_dir, 'master_ward_pending_complaintsnewest.csv')

master_df.to_csv(master_csv_path, index=False)
print(f"Master CSV with all offense counts by ward saved to {master_csv_path}")

print("All offense maps, tables, and CSV files have been generated successfully!")

In [ ]:
#Code for GVP identification
import pandas as pd
import folium
import geopandas as gpd
from sklearn.cluster import DBSCAN
import numpy as np
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from tabulate import tabulate

# Load the CSV file with a specified encoding
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
complaints_df = pd.read_csv(csv_path, encoding='ISO-8859-1')

# Remove rows 2 to 57556
complaints_df = complaints_df.drop(complaints_df.index[1:57556])

# Split latitude and longitude into separate columns
complaints_df[['Latitude', 'Longitude']] = complaints_df['Latitude & Longitude'].str.split(',', expand=True)
complaints_df['Latitude'] = complaints_df['Latitude'].astype(float)
complaints_df['Longitude'] = complaints_df['Longitude'].astype(float)

# Filter for "Dumping of Construction & Demolition Waste" offences
illegal_dumping_df = complaints_df[
    complaints_df['Offences'].isin([
        'Illegal dumping of Garbage on road sides/ vacant land',
        'Dumping of Construction & Demolition Waste'])].copy()

# Convert coordinates to radians for haversine calculation
illegal_dumping_df['Latitude_rad'] = np.radians(illegal_dumping_df['Latitude'])
illegal_dumping_df['Longitude_rad'] = np.radians(illegal_dumping_df['Longitude'])

# Apply DBSCAN with a 5-meter radius (approximately 0.000045 radians) and minimum samples of 1
coords = illegal_dumping_df[['Latitude_rad', 'Longitude_rad']].values
db = DBSCAN(eps=0.00000085, min_samples=1, metric='haversine').fit(coords)

# Add cluster labels to the DataFrame
illegal_dumping_df['Cluster'] = db.labels_

# Group by clusters and count occurrences
clustered_counts = illegal_dumping_df.groupby('Cluster').size().reset_index(name='Occurrences')

# Get the mean location for each cluster
cluster_centers = illegal_dumping_df.groupby('Cluster').agg({
    'Latitude': 'mean',
    'Longitude': 'mean',
    'Geo Location': 'first',  # Take the first occurrence of Geo Location as representative
}).reset_index()

# Merge the counts with the cluster centers
clustered_data = pd.merge(cluster_centers, clustered_counts, on='Cluster')

# Add a column to identify if the cluster has any pending status complaints
pending_status_by_cluster = illegal_dumping_df.groupby('Cluster')['Status'].apply(
    lambda x: 'Pending' in x.values).reset_index(name='Has_Pending')

# Merge this information with clustered_data
clustered_data = pd.merge(clustered_data, pending_status_by_cluster, on='Cluster')

# Filter for clusters that have pending status
pending_clusters = clustered_data[clustered_data['Has_Pending']].copy()

# Tabulate top 100 occurrences with pending status
top_100_occurrences = pending_clusters.sort_values(by='Occurrences', ascending=False).head(100)

# Normalize occurrences for colormap
norm = mcolors.Normalize(vmin=top_100_occurrences['Occurrences'].min(), vmax=top_100_occurrences['Occurrences'].max())
cmap = cm.get_cmap('YlOrRd')

# Load the shapefile
shapefile_path = r'C:\Users\Jeen priyee\Downloads\Delhi_Wards-SHP_Datameet\Delhi_Wards.shp'

wards_gdf = gpd.read_file(shapefile_path)

# Set CRS to EPSG:4326 if not already set
if wards_gdf.crs is None:
    wards_gdf.set_crs(epsg=4326, inplace=True)
else:
    wards_gdf.to_crs(epsg=4326, inplace=True)

# Create a folium map with a white background
m = folium.Map(
    location=[28.7041, 77.1025], 
    zoom_start=11,
    tiles="cartodbpositron",  # Use a white base map
    attr="CartoDB Positron"
)

# Create a white overlay for entire map area first
folium.Rectangle(
    bounds=[[wards_gdf.total_bounds[1], wards_gdf.total_bounds[0]], 
            [wards_gdf.total_bounds[3], wards_gdf.total_bounds[2]]],
    color='white',
    fill=True,
    fill_color='white',
    fill_opacity=1.0
).add_to(m)

# Add ward boundaries to the map
folium.GeoJson(
    wards_gdf,
    name='Delhi Wards',
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'black', 'weight': 1}
).add_to(m)

# Add circle markers for each of the top 100 clusters with occurrences count
for idx, row in top_100_occurrences.iterrows():
    cluster_complaints = illegal_dumping_df[illegal_dumping_df['Cluster'] == row['Cluster']]
    
    # Get pending complaints in this cluster
    pending_complaints = cluster_complaints[cluster_complaints['Status'] == 'Pending']
    
    resolve_images = ''.join([f"<img src='{img}' width='150' height='150'><br>" for img in cluster_complaints['Resolve Image'].dropna()])
    offence_images = ''.join([f"<img src='{img}' width='150' height='150'><br>" for img in cluster_complaints['Offence Image'].dropna()])
    color = mcolors.to_hex(cmap(norm(row['Occurrences'])))
    
    # Add pending status count to popup
    pending_count = len(pending_complaints)
    total_count = len(cluster_complaints)
    
    popup_content = (f"Occurrences: {row['Occurrences']}<br>"
                     f"Pending Complaints: {pending_count} of {total_count}<br>"
                     f"Location: {row['Geo Location']}<br>"
                     f"Date: {', '.join(cluster_complaints['Date and Time'].dropna())}<br>"
                     f"Status: {', '.join(cluster_complaints['Status'].dropna())}<br>"
                     f"Resolve Images:<br>{resolve_images}<br>"
                     f"Offence Images:<br>{offence_images}")
    
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=5 + row['Occurrences'] / 10,  # Increase radius based on occurrences
        color=color, fill=True, fill_color=color, fill_opacity=0.7,
        popup=folium.Popup(popup_content, max_width=300, max_height=600)).add_to(m)

# Add legend
legend_html = '''
<div style="position: fixed; 
    bottom: 50px; left: 50px; width: 250px; height: 150px; 
    border:2px solid grey; z-index:9999; font-size:14px;
    background-color:white;
    padding: 10px;
    ">
    <b>Occurrences Legend</b><br>
    <i style="background:#ffffb2; width: 20px; height: 20px; display: inline-block;"></i>&nbsp;Low (1-3)<br>
    <i style="background:#fd8d3c; width: 20px; height: 20px; display: inline-block;"></i>&nbsp;Medium (4-10)<br>
    <i style="background:#bd0026; width: 20px; height: 20px; display: inline-block;"></i>&nbsp;High (>10)<br>
    <br>
    <i>Note: Only showing locations with pending complaints</i>
</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))

# Remove the folium attribution and any other map elements except what we want
m.get_root().html.add_child(folium.Element('''
<style>
.leaflet-control-attribution {
    display: none;
}
.leaflet-control-zoom {
    display: none;
}
</style>
'''))

# Save the map to an HTML file to view it in a browser
m.save('delhi_pending_complaints_map.html')

# Display the top 10 locations and their occurrence numbers
top_10_locations = top_100_occurrences[['Occurrences', 'Geo Location', 'Has_Pending']].head(10)

# Enhanced tabulation output for top 10 locations
print("\nTop 10 Locations with Pending Complaints:")
print(tabulate(top_10_locations[['Occurrences', 'Geo Location']], headers=['Occurrences', 'Geo Location'], tablefmt='fancy_grid', showindex=False, stralign='center'))

# Display the map in the notebook (if supported)
m

In [ ]:
#Code for GVP identification
import pandas as pd
import folium
import geopandas as gpd
from sklearn.cluster import DBSCAN
import numpy as np
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from tabulate import tabulate

# Load the CSV file with a specified encoding
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
complaints_df = pd.read_csv(csv_path, encoding='ISO-8859-1')

# Remove rows 2 to 57556
complaints_df = complaints_df.drop(complaints_df.index[1:57556])

# Split latitude and longitude into separate columns
complaints_df[['Latitude', 'Longitude']] = complaints_df['Latitude & Longitude'].str.split(',', expand=True)
complaints_df['Latitude'] = complaints_df['Latitude'].astype(float)
complaints_df['Longitude'] = complaints_df['Longitude'].astype(float)

# 1. CREATE SAFE WORKING COPY
df = complaints_df.copy()

# Drop rows without valid coordinates
df = df.dropna(subset=['Latitude', 'Longitude'])

# 2. DBSCAN CLUSTERING (5 METERS)
df['lat_rad'] = np.radians(df['Latitude'])
df['lon_rad'] = np.radians(df['Longitude'])

coords = df[['lat_rad', 'lon_rad']].values

EARTH_RADIUS_M = 6_371_000
eps = 5 / EARTH_RADIUS_M  # 5 meters

db = DBSCAN(
    eps=eps,
    min_samples=1,
    metric='haversine'
).fit(coords)

df['Cluster'] = db.labels_

# 3. CITY-WIDE AREA ASSIGNMENT
# If you have a ward/zone column, replace 'City-wide' with that column
df['Area'] = 'City-wide'

# 4. KEEP ONLY RECURRING LOCATIONS (≥ 2 COMPLAINTS)
cluster_counts = (
    df.groupby('Cluster')['Compliant ID']
    .nunique()
    .reset_index(name='Count')
)

recurring_clusters = cluster_counts.loc[
    cluster_counts['Count'] >= 2, 'Cluster'
]

df_recurring = df[df['Cluster'].isin(recurring_clusters)].copy()

# 5. LOCATION-LEVEL SUMMARY
location_summary = (
    df_recurring
    .groupby('Cluster')
    .agg(
        Geo_Location=('Geo Location', 'first'),
        Count=('Compliant ID', 'nunique'),
        Latitude=('Latitude', 'mean'),
        Longitude=('Longitude', 'mean'),
        Area=('Area', 'first')
    )
    .reset_index()
)

# 6. OFFENCE TYPES WITH COUNTS (HUMAN-READABLE)
offence_counts = (
    df_recurring
    .groupby(['Cluster', 'Offences'])['Compliant ID']
    .nunique()
    .reset_index(name='cnt')
)

offence_text = (
    offence_counts
    .assign(
        offence_str=lambda x:
        x['Offences'] + ' (' + x['cnt'].astype(str) + ')'
    )
    .groupby('Cluster')['offence_str']
    .apply(', '.join)
    .reset_index(name='Offences')
)

# 7. FINAL TABLE (EXACT REQUIRED COLUMNS)
final_table = (
    location_summary
    .merge(offence_text, on='Cluster', how='left')
    [['Geo_Location', 'Count', 'Offences', 'Latitude', 'Longitude', 'Area']]
)

# 8. SORT, NUMBER HOTSPOTS, EXPORT
final_table_sorted = (
    final_table
    .sort_values(by='Count', ascending=False)
    .reset_index(drop=True)
)

final_table_sorted.insert(
    0,
    'Hotspot',
    ['Hotspot ' + str(i + 1) for i in range(len(final_table_sorted))]
)

download_path = os.path.join(
    os.path.expanduser("~"),
    "Downloads",
    "recurring_complaint_hotspots_citywide.xlsx"
)

final_table_sorted.to_excel(download_path, index=False)

# 9. DISPLAY OUTPUT
pd.set_option("display.max_rows", None)
print(final_table_sorted)

print("\nExcel saved at:")
print(download_path)


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
from sklearn.cluster import DBSCAN
from shapely.geometry import Point

# --------------------------------------------------
# 1. LOAD CSV
# --------------------------------------------------
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
complaints_df = pd.read_csv(csv_path, encoding='ISO-8859-1')

# Split Latitude & Longitude
complaints_df[['Latitude', 'Longitude']] = (
    complaints_df['Latitude & Longitude']
    .str.split(',', expand=True)
)

complaints_df['Latitude'] = complaints_df['Latitude'].astype(float)
complaints_df['Longitude'] = complaints_df['Longitude'].astype(float)

# Drop invalid coordinates
complaints_df = complaints_df.dropna(subset=['Latitude', 'Longitude'])

# --------------------------------------------------
# 2. DBSCAN CLUSTERING (5 METERS)
# --------------------------------------------------
complaints_df['lat_rad'] = np.radians(complaints_df['Latitude'])
complaints_df['lon_rad'] = np.radians(complaints_df['Longitude'])

coords = complaints_df[['lat_rad', 'lon_rad']].values

EARTH_RADIUS_M = 6_371_000
eps = 5 / EARTH_RADIUS_M  # 5 meters

db = DBSCAN(
    eps=eps,
    min_samples=1,
    metric='haversine'
).fit(coords)

complaints_df['Cluster'] = db.labels_

# --------------------------------------------------
# 3. KEEP ONLY RECURRING LOCATIONS (>= 2 COMPLAINTS)
# --------------------------------------------------
cluster_counts = (
    complaints_df
    .groupby('Cluster')
    .size()
    .reset_index(name='Count')
)

recurring_clusters = cluster_counts.loc[
    cluster_counts['Count'] >= 2, 'Cluster'
]

df_recurring = complaints_df[
    complaints_df['Cluster'].isin(recurring_clusters)
].copy()

# --------------------------------------------------
# 4. LOCATION-LEVEL SUMMARY
# --------------------------------------------------
location_summary = (
    df_recurring
    .groupby('Cluster')
    .agg(
        Count=('Cluster', 'size'),
        Latitude=('Latitude', 'mean'),
        Longitude=('Longitude', 'mean')
    )
    .reset_index()
)

# --------------------------------------------------
# 5. CLASSIFY HOTSPOT INTENSITY
# --------------------------------------------------
def classify(count):
    if count <= 3:
        return "Low (1–3)"
    elif count <= 10:
        return "Medium (4–10)"
    else:
        return "High (>10)"

location_summary['Category'] = location_summary['Count'].apply(classify)

color_map = {
    "Low (1–3)": "#fee08b",
    "Medium (4–10)": "#fc8d59",
    "High (>10)": "#d73027"
}

# --------------------------------------------------
# 6. LOAD DELHI WARD SHAPEFILE
# --------------------------------------------------
shp_path = r"C:\Users\Jeen priyee\Downloads\Delhi_Wards-SHP_Datameet\Delhi_Wards.shp"
delhi_gdf = gpd.read_file(shp_path).to_crs(epsg=4326)

# --------------------------------------------------
# 7. CREATE FOLIUM MAP
# --------------------------------------------------
m = folium.Map(
    location=[28.6139, 77.2090],
    zoom_start=11,
    tiles="cartodbpositron"
)

# Add Delhi wards boundary
folium.GeoJson(
    delhi_gdf,
    name="Delhi Wards",
    style_function=lambda x: {
        "fillColor": "none",
        "color": "black",
        "weight": 1
    }
).add_to(m)

# --------------------------------------------------
# 8. ADD HOTSPOT CIRCLES
# --------------------------------------------------
for _, row in location_summary.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=4 + row['Count'] * 0.4,
        color=color_map[row['Category']],
        fill=True,
        fill_color=color_map[row['Category']],
        fill_opacity=0.8,
        popup=(
            f"<b>Hotspot</b><br>"
            f"Complaints: {row['Count']}<br>"
            f"Category: {row['Category']}"
        )
    ).add_to(m)

# --------------------------------------------------
# 9. ADD LEGEND
# --------------------------------------------------
legend_html = """
<div style="
position: fixed;
bottom: 30px;
right: 30px;
width: 190px;
background-color: white;
border:2px solid grey;
z-index:9999;
font-size:14px;
padding: 10px;
">
<b>Count of Occurrence</b><br>
<span style="background:#fee08b;width:12px;height:12px;display:inline-block;"></span> Low (1–3)<br>
<span style="background:#fc8d59;width:12px;height:12px;display:inline-block;"></span> Medium (4–10)<br>
<span style="background:#d73027;width:12px;height:12px;display:inline-block;"></span> High (&gt;10)
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# --------------------------------------------------
# 10. SAVE MAP
# --------------------------------------------------
output_path = r"C:\Users\Jeen priyee\Downloads\Delhi_Complaint_Hotspots.html"
m.save(output_path)

print("Hotspot map saved at:")
print(output_path)


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
from sklearn.cluster import DBSCAN
from IPython.display import display

# --------------------------------------------------
# 1. LOAD CSV
# --------------------------------------------------
csv_path = r"C:\Users\Jeen priyee\Downloads\gda_new_2026\gda_updated_15jan2026.csv"
complaints_df = pd.read_csv(csv_path, encoding='ISO-8859-1')

complaints_df[['Latitude', 'Longitude']] = (
    complaints_df['Latitude & Longitude']
    .str.split(',', expand=True)
)

complaints_df['Latitude'] = complaints_df['Latitude'].astype(float)
complaints_df['Longitude'] = complaints_df['Longitude'].astype(float)

complaints_df = complaints_df.dropna(subset=['Latitude', 'Longitude'])

# --------------------------------------------------
# 1A. FILTER: ONLY DUMPING-RELATED OFFENCES (KEY EDIT)
# --------------------------------------------------
dumping_offences = [
    'Illegal dumping of Garbage on road sides/ vacant land',
    'Dumping of Construction & Demolition Waste'
]

complaints_df = complaints_df[
    complaints_df['Offences'].isin(dumping_offences)
].copy()

# --------------------------------------------------
# 2. DBSCAN CLUSTERING (5 METERS)
# --------------------------------------------------
complaints_df['lat_rad'] = np.radians(complaints_df['Latitude'])
complaints_df['lon_rad'] = np.radians(complaints_df['Longitude'])

coords = complaints_df[['lat_rad', 'lon_rad']].values

EARTH_RADIUS_M = 6_371_000
eps = 5 / EARTH_RADIUS_M

db = DBSCAN(
    eps=eps,
    min_samples=1,
    metric='haversine'
).fit(coords)

complaints_df['Cluster'] = db.labels_

# --------------------------------------------------
# 3. RECURRING LOCATIONS (>=2)
# --------------------------------------------------
cluster_counts = (
    complaints_df
    .groupby('Cluster')
    .size()
    .reset_index(name='Count')
)

recurring_clusters = cluster_counts.loc[
    cluster_counts['Count'] >= 2, 'Cluster'
]

df_recurring = complaints_df[
    complaints_df['Cluster'].isin(recurring_clusters)
].copy()


# --------------------------------------------------
# 4. LOCATION SUMMARY
# --------------------------------------------------
location_summary = (
    df_recurring
    .groupby('Cluster')
    .agg(
        Count=('Cluster', 'size'),
        Latitude=('Latitude', 'mean'),
        Longitude=('Longitude', 'mean')
    )
    .reset_index()
)

# --------------------------------------------------
# 5. CLASSIFY INTENSITY
# --------------------------------------------------
def classify(count):
    if count <= 3:
        return "Low (1–3)"
    elif count <= 10:
        return "Medium (4–10)"
    else:
        return "High (>10)"

location_summary['Category'] = location_summary['Count'].apply(classify)

color_map = {
    "Low (1–3)": "#fee08b",
    "Medium (4–10)": "#fc8d59",
    "High (>10)": "#d73027"
}

# --------------------------------------------------
# 6. LOAD DELHI WARDS
# --------------------------------------------------
shp_path = r"C:\Users\Jeen priyee\Downloads\Delhi_Wards-SHP_Datameet\Delhi_Wards.shp"
delhi_gdf = gpd.read_file(shp_path).to_crs(epsg=4326)

# --------------------------------------------------
# 7. CREATE MAP (INLINE)
# --------------------------------------------------
m = folium.Map(
    location=[28.6139, 77.2090],
    zoom_start=11,
    tiles="cartodbpositron"
)

folium.GeoJson(
    delhi_gdf,
    style_function=lambda x: {
        "fillColor": "none",
        "color": "black",
        "weight": 1
    }
).add_to(m)

# --------------------------------------------------
# 8. ADD HOTSPOTS
# --------------------------------------------------
for _, row in location_summary.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=4 + row['Count'] * 0.4,
        color=color_map[row['Category']],
        fill=True,
        fill_color=color_map[row['Category']],
        fill_opacity=0.8,
        popup=(
            f"<b>Hotspot</b><br>"
            f"Complaints: {row['Count']}<br>"
            f"Category: {row['Category']}"
        )
    ).add_to(m)

# --------------------------------------------------
# 9. LEGEND
# --------------------------------------------------
legend_html = """
<div style="
position: fixed;
bottom: 30px;
right: 30px;
width: 190px;
background-color: white;
border:2px solid grey;
z-index:9999;
font-size:14px;
padding: 10px;
">
<b>Count of Occurrence</b><br>
<span style="background:#fee08b;width:12px;height:12px;display:inline-block;"></span> Low (1–3)<br>
<span style="background:#fc8d59;width:12px;height:12px;display:inline-block;"></span> Medium (4–10)<br>
<span style="background:#d73027;width:12px;height:12px;display:inline-block;"></span> High (&gt;10)
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# --------------------------------------------------
# 10. DISPLAY MAP IN NOTEBOOK
# --------------------------------------------------
display(m)
